<a href="https://colab.research.google.com/github/1kaiser/Media-Segment-Depth-MLP/blob/main/STRING_3D_ViTTiny_Elayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## string 3d ViT multilayer attention experiment

### folder creations

In [ ]:
!pip install -q grain opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.4/499.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 1.9 MB/s eta 0:00:00


In [ ]:
%cd /content/

/content


In [ ]:
import os
import shutil
import glob


# Create the project structure
PROJECT_DIR = "vision_transformer_depth"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Project directories are set up successfully!")

✅ Project directories are set up successfully!


### model code

In [ ]:
%cd /content/

/content


In [ ]:
%%writefile vision_transformer_depth/string3d_experiment_pipeline.py

import os
import sys
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob

import jax
import jax.numpy as jnp
from jax import random, jit, value_and_grad

import flax.linen as nn
import optax
import grain
from einops import rearrange
from tqdm import trange

# --- SECTION 2: DATA LOADING (GRAIN) ---
class DepthDataSource:
    def __init__(self, data_dir="."):
        self.pairs = [(f, os.path.join(data_dir, "depth_maps", os.path.basename(f))) for f in glob.glob(os.path.join(data_dir, "input_frames", "*.png")) if os.path.exists(os.path.join(data_dir, "depth_maps", os.path.basename(f)))]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        inp_path, dep_path = self.pairs[idx]
        inp_rgb = cv2.cvtColor(cv2.imread(inp_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(dep_path, cv2.IMREAD_GRAYSCALE)
        inp_rgb, dep = inp_rgb.astype(np.float32)/255.0, dep.astype(np.float32)/255.0
        h, w, _ = inp_rgb.shape
        y, x = max(0, (h - 224) // 2), max(0, (w - 224) // 2)
        inp_cropped, dep_cropped = inp_rgb[y:y+224, x:x+224], dep[y:y+224, x:x+224]
        return {'image': inp_cropped, 'depth': dep_cropped[..., None]}

def create_data_iterator(data_dir, batch_size, seed=42):
    source = DepthDataSource(data_dir=data_dir)
    if not source.pairs: return iter([])
    dataset = (grain.MapDataset.source(source).shuffle(seed=seed).batch(batch_size, drop_remainder=True).repeat().to_iter_dataset())
    return iter(dataset)

# --- SECTION 3: MODEL ARCHITECTURE (FLAX) ---
class StringPositionEmbedding3D(nn.Module):
    embed_dim: int
    @nn.compact
    def __call__(self, x, x_coords, y_coords, z_coords):
        S = self.param('S_cayley', nn.initializers.normal(0.01), (self.embed_dim, self.embed_dim))
        S_antisym, I = (S - S.T) / 2.0, jnp.eye(self.embed_dim, dtype=x.dtype)
        P = jnp.linalg.solve(I + S_antisym, I - S_antisym)
        x_transformed = jnp.matmul(x, P.T)
        freqs = 1.0 / (10000 ** (jnp.arange(0, self.embed_dim // 2, dtype=jnp.float32) * 2 / self.embed_dim))
        angles = jnp.einsum('i,j->ij', x_coords, freqs) + jnp.einsum('i,j->ij', y_coords, freqs) + jnp.einsum('i,j->ij', z_coords, freqs)
        cos_vals, sin_vals = jnp.repeat(jnp.cos(angles), 2, axis=-1), jnp.repeat(jnp.sin(angles), 2, axis=-1)
        x1, x2 = jnp.split(x_transformed, 2, axis=-1)
        return x_transformed * cos_vals[None, None, :, :] + jnp.concatenate([-x2, x1], axis=-1) * sin_vals[None, None, :, :]

class String3DViTAttention(nn.Module):
    num_heads: int; embed_dim: int; patch_size: int = 16
    @nn.compact
    def __call__(self, x, depth_map):
        b, s, e = x.shape; head_dim = self.embed_dim // self.num_heads
        string_encoder = StringPositionEmbedding3D(embed_dim=head_dim, name="string_3d_encoder")
        qkv = nn.Dense(features=self.embed_dim * 3, use_bias=False, name="qkv")(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        q, k, v = [q.reshape(b, s, self.num_heads, head_dim).transpose(0, 2, 1, 3) for q in (q, k, v)]
        n_patches_per_dim = int(np.sqrt(s - 1))
        y_grid, x_grid = jnp.meshgrid(jnp.arange(n_patches_per_dim), jnp.arange(n_patches_per_dim), indexing='ij')
        x_coords, y_coords = jnp.concatenate([jnp.zeros(1), x_grid.flatten() + 1]), jnp.concatenate([jnp.zeros(1), y_grid.flatten() + 1])
        patch_depths = jnp.mean(rearrange(depth_map, 'b (h p1) (w p2) c -> b (h w) p1 p2 c', p1=self.patch_size, p2=self.patch_size), axis=(2,3)).squeeze(-1)
        z_coords_BL = jnp.concatenate([jnp.zeros((b, 1)), patch_depths], axis=1)
        z_coords_mean = jnp.mean(z_coords_BL, axis=0)
        q_final, k_final = string_encoder(q, x_coords, y_coords, z_coords_mean), string_encoder(k, x_coords, y_coords, z_coords_mean)
        scores = (q_final @ k_final.transpose(0, 1, 3, 2)) / jnp.sqrt(head_dim)
        attn = nn.softmax(scores, axis=-1)
        output = (attn @ v).transpose(0, 2, 1, 3).reshape(b, s, e)
        return nn.Dense(features=self.embed_dim, name="out_proj")(output)

class ViTTransformerBlock(nn.Module):
    embed_dim: int; num_heads: int
    @nn.compact
    def __call__(self, x, depth_map):
        attn_out = String3DViTAttention(embed_dim=self.embed_dim, num_heads=self.num_heads, name="attention")(nn.LayerNorm()(x), depth_map)
        x = x + attn_out
        mlp_out = nn.Dense(features=self.embed_dim * 4)(nn.LayerNorm()(x))
        mlp_out = nn.gelu(mlp_out)
        mlp_out = nn.Dense(features=self.embed_dim)(mlp_out)
        return x + mlp_out

class DepthHead(nn.Module):
    @nn.compact
    def __call__(self, patch_features):
        n_patches_per_dim = int(patch_features.shape[1] ** 0.5)
        x = rearrange(patch_features, 'b (h w) c -> b h w c', h=n_patches_per_dim)
        x = nn.ConvTranspose(features=128, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=64, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=32, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=16, kernel_size=(2, 2), strides=(2, 2))(x)
        return nn.Conv(features=1, kernel_size=(1, 1))(x)

class ViTForDepth_String3D(nn.Module):
    # --- *** NEW: Parameters are now instance attributes *** ---
    patch_size: int; num_layers: int; embed_dim: int; num_heads: int
    @nn.compact
    def __call__(self, image, depth_map, train: bool = True):
        patches = nn.Conv(features=self.embed_dim, kernel_size=(self.patch_size, self.patch_size), strides=(self.patch_size, self.patch_size))(image)
        patches = patches.reshape(patches.shape[0], -1, self.embed_dim)
        cls_token = self.param('cls_token', nn.initializers.zeros, (1, 1, self.embed_dim))
        x = jnp.concatenate([jnp.tile(cls_token, (patches.shape[0], 1, 1)), patches], axis=1)
        x = nn.Dropout(rate=0.1, deterministic=not train)(x)
        for i in range(self.num_layers):
            x = ViTTransformerBlock(embed_dim=self.embed_dim, num_heads=self.num_heads, name=f"layer_{i}")(x, depth_map)
        return DepthHead(name="depth_head")(nn.LayerNorm()(x[:, 1:, :]))

# --- SECTION 4: TRAINING & CHECKPOINTING SCRIPT ---
def train_model(config):
    print(f"\n🚀 Starting Training: {config['num_layers']} Layers...")
    key, dropout_key = random.split(random.PRNGKey(config['seed']))
    data_iter = create_data_iterator(config['data_dir'], config['batch_size'], config['seed'])

    try: batch = next(data_iter)
    except StopIteration:
        print("Error: Data iterator is empty."); return None, []

    # --- *** NEW: Instantiate model with config values *** ---
    model = ViTForDepth_String3D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads']
    )
    params = model.init(key, batch['image'], batch['depth'], train=False)['params']
    optimizer = optax.adam(config['lr'])
    opt_state = optimizer.init(params)

    @jit
    def train_step(params, opt_state, batch, dropout_rng):
        images, gt_depths = batch['image'], batch['depth']
        def loss_fn(p):
            pred_depths = model.apply({'params': p}, images, gt_depths, train=True, rngs={'dropout': dropout_rng})
            return jnp.mean(jnp.abs(pred_depths.squeeze() - gt_depths.squeeze()))
        loss, grads = value_and_grad(loss_fn)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt_state, loss

    all_losses = []
    pbar = trange(config['steps'], desc=f"🔥 Training ({config['num_layers']} Layers)")
    for step in pbar:
        dropout_key, step_key = random.split(dropout_key)
        batch = next(data_iter)
        params, opt_state, loss = train_step(params, opt_state, batch, step_key)
        loss_val = loss.item()
        all_losses.append(loss_val)
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    with open(config['checkpoint_path'], "wb") as f: pickle.dump(params, f)
    print(f"✅ Training finished. Parameters saved to {config['checkpoint_path']}")
    return config['checkpoint_path'], all_losses

# --- SECTION 5: INFERENCE SCRIPT ---
def run_inference(config, sample_image_path):
    checkpoint_path = config['checkpoint_path']
    print(f"\n🧠 Running Inference for {config['num_layers']}-Layer Model...")
    with open(checkpoint_path, 'rb') as f: params = pickle.load(f)

    # --- *** NEW: Instantiate model with config values *** ---
    model = ViTForDepth_String3D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads']
    )

    img = cv2.cvtColor(cv2.imread(sample_image_path), cv2.COLOR_BGR2RGB)
    img = (img.astype(np.float32) / 255.0)[16:240, 16:240]
    img_batch = jnp.expand_dims(img, 0)

    @jit
    def predict(p, img):
        dummy_depth = jnp.zeros_like(img[:, :, :, :1])
        return model.apply({'params': p}, img, dummy_depth, train=False)

    predicted_depth = predict(params, img_batch).squeeze()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Depth Prediction using {config['num_layers']}-Layer ViT", fontsize=14)
    ax1.imshow(img); ax1.set_title("Input RGB"); ax1.axis('off')
    ax2.imshow(predicted_depth, cmap='magma'); ax2.set_title("Predicted Depth"); ax2.axis('off')

    output_name = os.path.join("outputs", f"prediction_{config['num_layers']}_layers.png")
    plt.savefig(output_name); plt.show()
    print(f"🖼️  Inference plot saved to: {output_name}")

# --- SECTION 6: MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    # --- *** NEW: Define the list of experiments to run *** ---
    layer_configs_to_test = [2, 4, 6, 12]
    all_results = {}

    for num_layers in layer_configs_to_test:
        config = {
            'seed': 42, 'data_dir': "./data", 'batch_size': 8,
            'steps': 5000, 'lr': 3e-4, # Fewer steps for quicker comparison
            'num_layers': num_layers,
            'embed_dim': 192,
            'num_heads': 3,
            'checkpoint_path': f"./checkpoints/string3d_model_{num_layers}_layers.pkl"
        }

        final_checkpoint, losses = train_model(config)
        all_results[f'{num_layers} Layers'] = losses

        if final_checkpoint:
            run_inference(config, sample_image_path="./data/input_frames/frame_0014.png")

    # --- *** NEW: Plot a final comparison of all loss curves *** ---
    plt.figure(figsize=(12, 7))
    for name, losses in all_results.items():
        plt.plot(losses, label=name)
    plt.title("Training Loss Comparison by Number of Layers")
    plt.xlabel("Training Step")
    plt.ylabel("L1 Loss")
    plt.legend()
    plt.grid(True)
    comparison_path = os.path.join("outputs", "loss_curve_comparison.png")
    plt.savefig(comparison_path)
    plt.show()
    print(f"📉 Final comparison plot saved to: {comparison_path}")

Writing vision_transformer_depth/string3d_experiment_pipeline.py


### frame dataset downloading

In [ ]:
%cd /content/

/content


In [ ]:
!wget -nc -q https://github.com/1kaiser/Media-Segment-Depth-MLP/releases/download/v0.2/input_depthMaps.zip
!unzip -o /content/input_depthMaps.zip > /dev/null 2>&1

In [ ]:
import os
import shutil

# Move data into the project directory
if os.path.exists('/content/input_frames'):
    dest_path = os.path.join(DATA_DIR, 'input_frames')
    if os.path.exists(dest_path): shutil.rmtree(dest_path)
    shutil.move('/content/input_frames', dest_path)
if os.path.exists('/content/depth_maps'):
    dest_path = os.path.join(DATA_DIR, 'depth_maps')
    if os.path.exists(dest_path): shutil.rmtree(dest_path)
    shutil.move('/content/depth_maps', dest_path)

print("✅ Project data are set up successfully!")

✅ Project data are set up successfully!


In [ ]:
# @title ### training cell
# Change to the project directory
%cd /content/vision_transformer_depth

# Run the full experiment pipeline
!python string3d_experiment_pipeline.py

/content/vision_transformer_depth
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(

🚀 Starting Training: 2 Layers...
🔥 Training (2 Layers):   0% 0/5000 [00:00<?, ?it/s]https://symbolize.stripped_domain/r/?trace=7afb66f07f5c,7afbce37e51f,7afb66e8dfa4,7afb66f0a653,7afb66a8fe80,7afb669fbed7,7afb61555696,7afb5ecba0d5,7afb5ecd5b9c,7afb5ecd66f4,7afb5b903a5e,7afb5b8fe84d,7afb5b8fd4c8,7afb5eb6e7fd,7afb5b891322,7afb5b83536c,7afb5b82669a,7afb77e4b8fc,7afb77e4c065,7afb7f33e4eb,7afb7f3397b6,7afb7e35077c,7afb7e35158d,7afb7e358ade,7afb801b8660,7afb84a5266b,5602e5&map= 
*** SIGTERM received by PID 1346 (TID 1346) on cpu 1 from PID 1170; stack trace: ***
PC: @     0x7a

### satellite datat input

In [ ]:

############# 3. DATA ACQUISITION (SHELL COMMANDS) #############

# 1. Download all necessary files
print("⬇️ Downloading dataset files...")
!rm -f /content/vision_transformer_depth/DigitalElevationModel_01.tif /content/folder01_part_aa /content/folder01_part_ab # Remove potentially corrupted files
!wget -nc --no-check-certificate -P /content/vision_transformer_depth https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/DigitalElevationModel_01.tif
!wget -nc --no-check-certificate -P /content https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/folder01_part_aa
!wget -nc --no-check-certificate -P /content https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/folder01_part_ab


# 2. Extract and organize the image data
print("📦 Extracting image files...")
!cat /content/folder01_part_* > /content/folder01.zip
!rm -rf /content/folder01_part_*
!mkdir -p /content/imagesfolder
!unzip -o /content/folder01.zip -d /content/imagesfolder > /dev/null 2>&1
!rm -rf /content/folder01.zip
!mv /content/imagesfolder/files/* /content/imagesfolder/ 2>/dev/null || true

# 3. Resample the DEM to match the satellite image resolution using the GDAL Python library
print("🗺️ Resampling DEM to match image resolution...")
import glob
from osgeo import gdal
import os

try:
    # Robustly find a sample tif file
    sample_tifs = glob.glob('/content/imagesfolder/*.tif')
    if not sample_tifs:
        raise FileNotFoundError("No .tif files found in /content/imagesfolder/ to use as a size reference.")

    sample_image_path = sample_tifs[0]
    print(f"Using '{os.path.basename(sample_image_path)}' as size reference.")

    # --- THE FIX: Use the GDAL Python API directly ---
    # Open the reference image to get its dimensions
    ref_ds = gdal.Open(sample_image_path)
    if ref_ds is None:
        raise RuntimeError(f"Failed to open reference image: {sample_image_path}")

    width = ref_ds.RasterXSize
    height = ref_ds.RasterYSize
    # Close the dataset
    ref_ds = None

    print(f"Detected image size: {width}x{height}. Resampling DEM...")

    # Perform the warp operation using the Python function
    gdal.Warp('/content/DEMx.tif', '/content/vision_transformer_depth/DigitalElevationModel_01.tif',
              width=width, height=height, resampleAlg='bilinear')

except Exception as e:
    print(f"⚠️ An error occurred during resampling. Error: {e}")
    print("Falling back to copying the original DEM.")
    # Check if the original DEM exists before attempting to copy
    if os.path.exists('/content/vision_transformer_depth/DigitalElevationModel_01.tif'):
        !cp /content/vision_transformer_depth/DigitalElevationModel_01.tif /content/DEMx.tif
    else:
        print("Original DEM not found. Cannot perform fallback copy.")


# --- Final Verification Step ---
if os.path.exists('/content/DEMx.tif'):
    print("✅ Data acquisition and preparation complete. DEMx.tif is ready.")
else:
    raise RuntimeError("FATAL ERROR: /content/DEMx.tif was not created. Cannot proceed.")

⬇️ Downloading dataset files...
--2025-09-27 17:11:04--  https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/DigitalElevationModel_01.tif
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/570306733/b4739f92-70b3-43e8-97e0-fac3d7a1be36?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-09-27T17%3A52%3A32Z&rscd=attachment%3B+filename%3DDigitalElevationModel_01.tif&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-09-27T16%3A52%3A08Z&ske=2025-09-27T17%3A52%3A32Z&sks=b&skv=2018-11-09&sig=Gck51941cHzo5xMJQpYaWQCXYgMCUs9QgRyBZqURNVU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1ODk5MzM2N

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


✅ Data acquisition and preparation complete. DEMx.tif is ready.


In [ ]:
import os
import shutil
import glob
import numpy as np
from osgeo import gdal
import cv2

print("--- Starting Satellite Data Preparation ---")

# Step 1: Clean data directory
print("🧹 Cleaning data directory...")
data_dir_path = '/content/vision_transformer_depth/data'
if os.path.exists(data_dir_path):
    # Use shell command for robust recursive deletion
    !rm -rf {data_dir_path}/*
    print(f"Cleared contents of {data_dir_path}")
else:
    print(f"Data directory not found, creating: {data_dir_path}")
os.makedirs(data_dir_path, exist_ok=True) # Ensure the data directory exists


def gdal_to_numpy(path: str, xoff=0, yoff=0, xsize=None, ysize=None) -> np.ndarray:
    """Safely opens a GDAL raster and converts a region to a NumPy array."""
    raster_ds = gdal.Open(path)
    if raster_ds is None:
        raise FileNotFoundError(f"GDAL failed to open or find the file at path: {path}")
    # Read the specified region, default to full raster if size is None
    return raster_ds.ReadAsArray(xoff, yoff, xsize, ysize).astype(np.float32)

# Normalization for the depth target
normalize_depth = lambda band: (band - band.min()) / (band.max() - band.min()) if band.max() > band.min() else np.zeros_like(band)

# Define paths and band numbers for RGB
satellite_image_dir = '/content/imagesfolder/'
dem_path = '/content/DEMx.tif'
# Use bands 3, 2, 1 for RGB (MODIS bands) ["b01", "b02", "b03", "b04", "b05", "b06", "b07"]
rgb_bands_numbers = ["b05", "b04", "b03"]


# Step 2: Determine dimensions
print("⬇️ Determining dimensions from satellite image bands and DEM...")

# Get dimensions from a sample RGB band and the DEM
try:
    # Find a sample file for the first RGB band
    sample_image_path_glob = glob.glob(os.path.join(satellite_image_dir, f'*{rgb_bands_numbers[0]}*.tif'))
    if not sample_image_path_glob:
         raise FileNotFoundError(f"No {rgb_bands_numbers[0]}.tif files found in {satellite_image_dir}")

    sample_image = gdal_to_numpy(sample_image_path_glob[0])
    depth_map = gdal_to_numpy(dem_path)

    img_h, img_w = sample_image.shape[:2]
    dep_h, dep_w = depth_map.shape[:2]

    # Use the minimum dimensions to avoid index errors
    min_height = min(img_h, dep_h)
    min_width = min(img_w, dep_w)

except Exception as e:
    print(f"Error determining dimensions: {e}")
    raise RuntimeError("FATAL ERROR: Could not determine image and DEM dimensions.")


image_size = 224 # Define the tile size

# Calculate the number of tiles in each dimension
num_rows = min_height // image_size
num_cols = min_width // image_size

print(f"Minimum dimensions: {min_width}x{min_height}")
print(f"Number of tiles: {num_cols} columns x {num_rows} rows = {num_cols * num_rows} tiles.")

# Create output directories for processed tiles
output_frames_dir = os.path.join(data_dir_path, 'input_frames')
output_depth_dir = os.path.join(data_dir_path, 'depth_maps')

os.makedirs(output_frames_dir, exist_ok=True)
os.makedirs(output_depth_dir, exist_ok=True)

# Step 3 & 4: Tile, process, and save data
print("Processing and saving tiles...")
tile_count = 0
if num_rows > 0 and num_cols > 0:
    for row_idx in range(num_rows):
        for col_idx in range(num_cols):
            row_start = row_idx * image_size
            col_start = col_idx * image_size

            # Extract and normalize depth tile
            try:
                depth_tile_array = gdal_to_numpy(dem_path, xoff=col_start, yoff=row_start, xsize=image_size, ysize=image_size)
                depth_tile_normalized = normalize_depth(depth_tile_array)
            except Exception as e:
                print(f"Error processing depth tile at ({row_idx}, {col_idx}): {e}. Skipping tile.")
                continue # Skip this tile if depth processing fails


            # Extract image tiles for each RGB band and stack them
            image_bands_data = []
            bands_available_for_rgb = True
            try:
                for band_num_str in rgb_bands_numbers:
                     band_files = glob.glob(os.path.join(satellite_image_dir, f'*{band_num_str}*.tif'))
                     if band_files:
                         band_data = gdal_to_numpy(band_files[0], xoff=col_start, yoff=row_start, xsize=image_size, ysize=image_size)
                         if band_data.shape != (image_size, image_size):
                             print(f"Warning: Band {band_num_str} tile has incorrect shape {band_data.shape} at ({row_idx}, {col_idx}). Expected ({image_size}, {image_size}). Skipping tile.")
                             bands_available_for_rgb = False
                             break
                         image_bands_data.append(band_data)
                     else:
                         print(f"Error: Could not find file for band {band_num_str} for tiling at ({row_idx}, {col_idx}). Skipping tile.")
                         bands_available_for_rgb = False
                         break # Stop if any required RGB band is missing
            except Exception as e:
                 print(f"Error processing image bands tile at ({row_idx}, {col_idx}): {e}. Skipping tile.")
                 bands_available_for_rgb = False # Ensure we skip the tile


            # Stack the bands to create the RGB image tile if all required bands were found and shaped correctly
            if bands_available_for_rgb:
                image_tile_array = np.stack(image_bands_data, axis=-1)

                # Save the processed tiles
                frame_filename = os.path.join(output_frames_dir, f'frame_{tile_count:04d}.png')
                depth_filename = os.path.join(output_depth_dir, f'frame_{tile_count:04d}.png')

                # Convert numpy arrays to uint8 for saving as images
                # Scale image bands to 0-255 for uint8 saving.
                image_tile_scaled = (image_tile_array - image_tile_array.min()) / (image_tile_array.max() - image_tile_array.min()) * 255 if image_tile_array.max() > image_tile_array.min() else np.zeros_like(image_tile_array)
                image_tile_uint8 = image_tile_scaled.astype(np.uint8)

                # Scale depth tile to 0-255
                depth_tile_uint8 = (depth_tile_normalized * 255).astype(np.uint8)

                # Ensure the image tile has 3 channels for RGB
                if image_tile_uint8.shape[-1] != 3:
                    print(f"Warning: Final image tile for saving does not have 3 channels at ({row_idx}, {col_idx}). Shape: {image_tile_uint8.shape[-1]}. Skipping save.")
                    continue # Skip saving if not 3 channels

                cv2.imwrite(frame_filename, cv2.cvtColor(image_tile_uint8, cv2.COLOR_RGB2BGR)) # Save as BGR for OpenCV
                cv2.imwrite(depth_filename, depth_tile_uint8)

                tile_count += 1
            else:
                print(f"Skipping tile saving at ({row_idx}, {col_idx}) due to processing errors.")

else:
    print("Warning: No tiles to process (minimum dimension is less than image_size).")

print(f"--- Finished processing {tile_count} tiles ---")

# Step 5: Verify data structure
print("\n--- Verifying Data Structure ---")
# Check if directories exist
print(f"Checking directory: {output_frames_dir}")
if os.path.exists(output_frames_dir):
    print(f"Directory exists: {output_frames_dir}")
    # List contents of input_frames
    print(f"\nContents of {output_frames_dir}:")
    frames_list = os.listdir(output_frames_dir)
    print(frames_list[:10]) # Print only the first 10 files to avoid flooding the output
    if len(frames_list) > 10: print(f"... and {len(frames_list) - 10} more files.")
else:
    print(f"Directory does not exist: {output_frames_dir}")

print(f"\nChecking directory: {output_depth_dir}")
if os.path.exists(output_depth_dir):
    print(f"Directory exists: {output_depth_dir}")
    # List contents of depth_maps
    print(f"\nContents of {output_depth_dir}:")
    depth_list = os.listdir(output_depth_dir)
    print(depth_list[:10]) # Print only the first 10 files
    if len(depth_list) > 10: print(f"... and {len(depth_list) - 10} more files.")
else:
    print(f"Directory does not exist: {output_depth_dir}")

# Count the number of files in each directory
num_image_files = len([name for name in os.listdir(output_frames_dir) if os.path.isfile(os.path.join(output_frames_dir, name))]) if os.path.exists(output_frames_dir) else 0
num_depth_files = len([name for name in os.listdir(output_depth_dir) if os.path.isfile(os.path.join(output_depth_dir, name))]) if os.path.exists(output_depth_dir) else 0


print(f"\nNumber of image files found: {num_image_files}")
print(f"Number of depth files found: {num_depth_files}")

# Verify that the number of files match
if num_image_files == num_depth_files and num_image_files > 0:
    print("✅ Verification successful: Number of image files matches the number of depth files.")
else:
    print("❌ Verification failed: The number of image and depth files do not match or no files were found.")

print("--- Satellite Data Preparation Complete ---")

--- Starting Satellite Data Preparation ---
🧹 Cleaning data directory...
Cleared contents of /content/vision_transformer_depth/data
⬇️ Determining dimensions from satellite image bands and DEM...
Minimum dimensions: 15171x237
Number of tiles: 67 columns x 1 rows = 67 tiles.
Processing and saving tiles...
--- Finished processing 67 tiles ---

--- Verifying Data Structure ---
Checking directory: /content/vision_transformer_depth/data/input_frames
Directory exists: /content/vision_transformer_depth/data/input_frames

Contents of /content/vision_transformer_depth/data/input_frames:
['frame_0015.png', 'frame_0003.png', 'frame_0065.png', 'frame_0059.png', 'frame_0058.png', 'frame_0005.png', 'frame_0054.png', 'frame_0004.png', 'frame_0000.png', 'frame_0010.png']
... and 57 more files.

Checking directory: /content/vision_transformer_depth/data/depth_maps
Directory exists: /content/vision_transformer_depth/data/depth_maps

Contents of /content/vision_transformer_depth/data/depth_maps:
['frame_

In [ ]:
# @title ### training cell
# Change to the project directory
%cd /content/vision_transformer_depth

# Run the full experiment pipeline
!python string3d_experiment_pipeline.py

/content/vision_transformer_depth
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(

🚀 Starting Training: 2 Layers...
🔥 Training (2 Layers):   0% 0/5000 [00:18<?, ?it/s]
Traceback (most recent call last):
  File "/content/vision_transformer_depth/string3d_experiment_pipeline.py", line 208, in <module>
    final_checkpoint, losses = train_model(config)
                               ^^^^^^^^^^^^^^^^^^^
  File "/content/vision_transformer_depth/string3d_experiment_pipeline.py", line 149, in train_model
    params, opt_state, loss = train_step(params, opt_state, batch, step_key)
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

In [ ]:
# @title ### nnx_string3d_experiment_pipeline
%%writefile /content/vision_transformer_depth/nnx_string3d_experiment_pipeline.py
"""
Vision Transformer with String3D Position Encoding for Depth Estimation
CORRECTED VERSION - Removes circular dependency on depth
"""

import os
import sys
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
from typing import Optional, Tuple, Dict, List

import jax
import jax.numpy as jnp
from jax import random, jit

import flax.nnx as nnx
import optax
import grain
from einops import rearrange
from tqdm import trange


# ============================================================================
# SECTION 1: DATA LOADING (GRAIN)
# ============================================================================

class DepthDataSource:
    """Data source for loading RGB images and corresponding depth maps."""

    def __init__(self, data_dir: str = "."):
        """Initialize data source with directory containing input frames and depth maps."""
        self.pairs = [
            (f, os.path.join(data_dir, "depth_maps", os.path.basename(f)))
            for f in glob.glob(os.path.join(data_dir, "input_frames", "*.png"))
            if os.path.exists(os.path.join(data_dir, "depth_maps", os.path.basename(f)))
        ]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, jnp.ndarray]:
        """Load and preprocess image-depth pair at given index."""
        inp_path, dep_path = self.pairs[idx]

        # Load images
        inp_rgb = cv2.cvtColor(cv2.imread(inp_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(dep_path, cv2.IMREAD_GRAYSCALE)

        # Normalize to [0, 1]
        inp_rgb = inp_rgb.astype(np.float32) / 255.0
        dep = dep.astype(np.float32) / 255.0

        # Center crop to 224x224
        h, w, _ = inp_rgb.shape
        y = max(0, (h - 224) // 2)
        x = max(0, (w - 224) // 2)

        inp_cropped = inp_rgb[y:y+224, x:x+224]
        dep_cropped = dep[y:y+224, x:x+224]

        return {
            'image': inp_cropped,
            'depth': dep_cropped[..., None]
        }


def create_data_iterator(data_dir: str, batch_size: int, seed: int = 42):
    """Create a data iterator using Grain for efficient data loading."""
    source = DepthDataSource(data_dir=data_dir)

    if not source.pairs:
        return iter([])

    dataset = (
        grain.MapDataset.source(source)
        .shuffle(seed=seed)
        .batch(batch_size, drop_remainder=True)
        .repeat()
        .to_iter_dataset()
    )

    return iter(dataset)


# ============================================================================
# SECTION 2: MODEL ARCHITECTURE (FLAX NNX)
# ============================================================================

class StringPositionEmbedding2D(nnx.Module):
    """2D string-based position encoding using orthogonal transformations.

    This version only uses spatial (x, y) coordinates, not depth.
    """

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize string position embedding module."""
        self.embed_dim = embed_dim

        # Cayley parameterization for orthogonal matrix
        self.S_cayley = nnx.Param(
            nnx.initializers.normal(stddev=0.01)(
                rngs.params(), (embed_dim, embed_dim)
            )
        )

    def __call__(
        self,
        x: jnp.ndarray,
        x_coords: jnp.ndarray,
        y_coords: jnp.ndarray
    ) -> jnp.ndarray:
        """Apply 2D string position encoding to input."""
        # Construct orthogonal matrix via Cayley transform
        S_antisym = (self.S_cayley.value - self.S_cayley.value.T) / 2.0
        I = jnp.eye(self.embed_dim, dtype=x.dtype)
        P = jnp.linalg.solve(I + S_antisym, I - S_antisym)

        # Apply orthogonal transformation
        x_transformed = jnp.matmul(x, P.T)

        # Generate frequency encodings for 2D coordinates
        freqs = 1.0 / (10000 ** (
            jnp.arange(0, self.embed_dim // 2, dtype=jnp.float32) * 2 / self.embed_dim
        ))

        # Combine x and y coordinates
        angles = (
            jnp.einsum('i,j->ij', x_coords, freqs) +
            jnp.einsum('i,j->ij', y_coords, freqs)
        )

        cos_vals = jnp.repeat(jnp.cos(angles), 2, axis=-1)
        sin_vals = jnp.repeat(jnp.sin(angles), 2, axis=-1)

        # Apply rotational encoding
        x1, x2 = jnp.split(x_transformed, 2, axis=-1)
        encoded = (
            x_transformed * cos_vals[None, None, :, :] +
            jnp.concatenate([-x2, x1], axis=-1) * sin_vals[None, None, :, :]
        )

        return encoded


class String2DViTAttention(nnx.Module):
    """Multi-head attention with 2D string position encoding."""

    def __init__(
        self,
        num_heads: int,
        embed_dim: int,
        patch_size: int,
        rngs: nnx.Rngs
    ):
        """Initialize attention module."""
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.head_dim = embed_dim // num_heads

        # Attention layers
        self.qkv = nnx.Linear(embed_dim, embed_dim * 3, use_bias=False, rngs=rngs)
        self.out_proj = nnx.Linear(embed_dim, embed_dim, rngs=rngs)

        # String position encoder (2D only)
        self.string_encoder = StringPositionEmbedding2D(
            embed_dim=self.head_dim,
            rngs=rngs
        )

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        """Apply attention with 2D position encoding."""
        b, s, e = x.shape

        # Generate QKV
        qkv = self.qkv(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape for multi-head attention
        q = q.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        # Generate spatial coordinates
        n_patches_per_dim = int(np.sqrt(s - 1))  # Excluding CLS token
        y_grid, x_grid = jnp.meshgrid(
            jnp.arange(n_patches_per_dim),
            jnp.arange(n_patches_per_dim),
            indexing='ij'
        )

        # Include CLS token position (0, 0)
        x_coords = jnp.concatenate([jnp.zeros(1), x_grid.flatten() + 1])
        y_coords = jnp.concatenate([jnp.zeros(1), y_grid.flatten() + 1])

        # Normalize coordinates to [-1, 1]
        x_coords = (x_coords / n_patches_per_dim) * 2 - 1
        y_coords = (y_coords / n_patches_per_dim) * 2 - 1

        # Apply string encoding to queries and keys
        q_encoded = self.string_encoder(q, x_coords, y_coords)
        k_encoded = self.string_encoder(k, x_coords, y_coords)

        # Compute attention scores
        scores = (q_encoded @ k_encoded.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)
        attn_weights = nnx.softmax(scores, axis=-1)

        # Apply attention and reshape
        output = attn_weights @ v
        output = output.transpose(0, 2, 1, 3).reshape(b, s, e)

        return self.out_proj(output)


class ViTTransformerBlock(nnx.Module):
    """Transformer block with string-based 2D attention."""

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        patch_size: int,
        rngs: nnx.Rngs
    ):
        """Initialize transformer block."""
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Layers
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.attention = String2DViTAttention(
            num_heads=num_heads,
            embed_dim=embed_dim,
            patch_size=patch_size,
            rngs=rngs
        )
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.mlp_fc1 = nnx.Linear(embed_dim, embed_dim * 4, rngs=rngs)
        self.mlp_fc2 = nnx.Linear(embed_dim * 4, embed_dim, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        """Forward pass through transformer block."""
        # Attention with residual
        attn_out = self.attention(self.norm1(x))
        attn_out = self.dropout(attn_out, deterministic=not train)
        x = x + attn_out

        # MLP with residual
        mlp_out = self.norm2(x)
        mlp_out = self.mlp_fc1(mlp_out)
        mlp_out = nnx.gelu(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        mlp_out = self.mlp_fc2(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        x = x + mlp_out

        return x


class DepthHead(nnx.Module):
    """Decoder head for depth map reconstruction."""

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize depth decoder head."""
        self.embed_dim = embed_dim

        # Project to initial feature dimension
        self.proj = nnx.Linear(embed_dim, 256, rngs=rngs)

        # Upsampling layers with skip connections
        self.upsample1 = nnx.ConvTranspose(
            in_features=256,
            out_features=128,
            kernel_size=(2, 2),
            strides=(2, 2),
            rngs=rngs
        )
        self.conv1 = nnx.Conv(
            in_features=128,
            out_features=128,
            kernel_size=(3, 3),
            padding=1,
            rngs=rngs
        )

        self.upsample2 = nnx.ConvTranspose(
            in_features=128,
            out_features=64,
            kernel_size=(2, 2),
            strides=(2, 2),
            rngs=rngs
        )
        self.conv2 = nnx.Conv(
            in_features=64,
            out_features=64,
            kernel_size=(3, 3),
            padding=1,
            rngs=rngs
        )

        self.upsample3 = nnx.ConvTranspose(
            in_features=64,
            out_features=32,
            kernel_size=(2, 2),
            strides=(2, 2),
            rngs=rngs
        )
        self.conv3 = nnx.Conv(
            in_features=32,
            out_features=32,
            kernel_size=(3, 3),
            padding=1,
            rngs=rngs
        )

        self.upsample4 = nnx.ConvTranspose(
            in_features=32,
            out_features=16,
            kernel_size=(2, 2),
            strides=(2, 2),
            rngs=rngs
        )
        self.conv4 = nnx.Conv(
            in_features=16,
            out_features=16,
            kernel_size=(3, 3),
            padding=1,
            rngs=rngs
        )

        # Final convolution to single channel depth
        self.final_conv = nnx.Conv(
            in_features=16,
            out_features=1,
            kernel_size=(1, 1),
            rngs=rngs
        )

    def __call__(self, patch_features: jnp.ndarray) -> jnp.ndarray:
        """Decode patch features to depth map."""
        # Project features
        x = self.proj(patch_features)

        # Reshape patches to spatial grid (14x14 for 224x224 image with patch_size=16)
        n_patches_per_dim = int(np.sqrt(patch_features.shape[1]))
        x = rearrange(
            x,
            'b (h w) c -> b h w c',
            h=n_patches_per_dim,
            w=n_patches_per_dim
        )

        # Progressive upsampling with refinement
        x = self.upsample1(x)  # 14x14 -> 28x28
        x = nnx.relu(self.conv1(x))

        x = self.upsample2(x)  # 28x28 -> 56x56
        x = nnx.relu(self.conv2(x))

        x = self.upsample3(x)  # 56x56 -> 112x112
        x = nnx.relu(self.conv3(x))

        x = self.upsample4(x)  # 112x112 -> 224x224
        x = nnx.relu(self.conv4(x))

        # Final depth prediction with sigmoid for [0, 1] range
        x = self.final_conv(x)
        x = nnx.sigmoid(x)

        return x


class ViTForDepth_String2D(nnx.Module):
    """Vision Transformer with String2D encoding for depth estimation.

    CORRECTED: Only uses RGB images as input, no circular dependency on depth.
    """

    def __init__(
        self,
        patch_size: int,
        num_layers: int,
        embed_dim: int,
        num_heads: int,
        rngs: nnx.Rngs
    ):
        """Initialize ViT model."""
        self.patch_size = patch_size
        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Patch embedding
        self.patch_embed = nnx.Conv(
            in_features=3,
            out_features=embed_dim,
            kernel_size=(patch_size, patch_size),
            strides=(patch_size, patch_size),
            rngs=rngs
        )

        # CLS token (learnable)
        self.cls_token = nnx.Param(
            nnx.initializers.normal(stddev=0.02)(
                rngs.params(), (1, 1, embed_dim)
            )
        )

        # Dropout
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

        # Transformer blocks
        self.blocks = [
            ViTTransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                patch_size=patch_size,
                rngs=rngs
            )
            for _ in range(num_layers)
        ]

        # Final norm and depth head
        self.norm = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.depth_head = DepthHead(embed_dim, rngs=rngs)

    def __call__(self, image: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        """Forward pass through the model.

        Args:
            image: Input RGB image of shape (batch_size, height, width, 3)
            train: Whether in training mode

        Returns:
            Predicted depth map of shape (batch_size, height, width, 1)
        """
        # Extract patches
        patches = self.patch_embed(image)
        b, h, w, c = patches.shape
        patches = patches.reshape(b, h * w, c)

        # Add CLS token
        cls_tokens = jnp.tile(self.cls_token.value, (b, 1, 1))
        x = jnp.concatenate([cls_tokens, patches], axis=1)

        # Apply dropout
        x = self.dropout(x, deterministic=not train)

        # Process through transformer blocks
        for block in self.blocks:
            x = block(x, train=train)

        # Apply final norm
        x = self.norm(x)

        # Remove CLS token and decode depth
        patch_features = x[:, 1:, :]  # Remove CLS token
        depth_output = self.depth_head(patch_features)

        return depth_output


# ============================================================================
# SECTION 3: TRAINING & CHECKPOINTING
# ============================================================================

def compute_loss(pred_depth: jnp.ndarray, gt_depth: jnp.ndarray) -> jnp.ndarray:
    """Compute depth estimation loss.

    Uses a combination of L1 loss and gradient loss for better edge preservation.
    """
    # L1 loss
    l1_loss = jnp.mean(jnp.abs(pred_depth - gt_depth))

    # Gradient loss for edge preservation
    def gradient(x):
        # Compute gradients in x and y directions
        grad_x = x[:, 1:, :, :] - x[:, :-1, :, :]
        grad_y = x[:, :, 1:, :] - x[:, :, :-1, :]
        return grad_x, grad_y

    pred_grad_x, pred_grad_y = gradient(pred_depth)
    gt_grad_x, gt_grad_y = gradient(gt_depth)

    grad_loss = (
        jnp.mean(jnp.abs(pred_grad_x - gt_grad_x)) +
        jnp.mean(jnp.abs(pred_grad_y - gt_grad_y))
    )

    # Combined loss
    total_loss = l1_loss + 0.5 * grad_loss

    return total_loss


def train_model(config: Dict) -> Tuple[Optional[str], List[float]]:
    """Train the depth estimation model."""
    print(f"\n{'='*60}")
    print(f"🚀 Starting Training: {config['num_layers']} Layers")
    print(f"{'='*60}")

    # Initialize RNGs
    rngs = nnx.Rngs(config['seed'])

    # Create data iterator
    data_iter = create_data_iterator(
        config['data_dir'],
        config['batch_size'],
        config['seed']
    )

    # Get sample batch for initialization
    try:
        batch = next(data_iter)
    except StopIteration:
        print("Error: Data iterator is empty.")
        return None, []

    # Initialize model
    model = ViTForDepth_String2D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        rngs=rngs
    )

    # Initialize optimizer with learning rate schedule
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=config['lr'],
        warmup_steps=500,
        decay_steps=config['steps'],
        end_value=1e-6
    )

    optimizer = nnx.Optimizer(
        model,
        optax.chain(
            optax.clip_by_global_norm(1.0),  # Gradient clipping
            optax.adam(schedule)
        )
    )

    @nnx.jit
    def train_step(model, optimizer, batch):
        """Single training step."""
        images = batch['image']
        gt_depths = batch['depth']

        def loss_fn(model):
            pred_depths = model(images, train=True)
            return compute_loss(pred_depths, gt_depths)

        loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(grads)

        return loss

    # Training loop
    all_losses = []
    pbar = trange(
        config['steps'],
        desc=f"🔥 Training ({config['num_layers']} Layers)"
    )

    for step in pbar:
        batch = next(data_iter)
        loss = train_step(model, optimizer, batch)
        loss_val = float(loss)
        all_losses.append(loss_val)

        # Update progress bar
        pbar.set_postfix(loss=f"{loss_val:.4f}")

        # Log every 100 steps
        if step % 100 == 0 and step > 0:
            avg_loss = np.mean(all_losses[-100:])
            print(f"\n[Step {step}] Average Loss: {avg_loss:.4f}")

    # Save checkpoint
    checkpoint_dir = os.path.dirname(config['checkpoint_path'])
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Extract and save model state
    model_state = nnx.state(model)
    with open(config['checkpoint_path'], "wb") as f:
        pickle.dump(model_state, f)

    print(f"✅ Training complete! Model saved to {config['checkpoint_path']}")
    print(f"   Final loss: {all_losses[-1]:.4f}")
    print(f"   Average last 100 steps: {np.mean(all_losses[-100:]):.4f}")

    return config['checkpoint_path'], all_losses


# ============================================================================
# SECTION 4: INFERENCE
# ============================================================================

def run_inference(config: Dict, sample_image_path: str):
    """Run inference with trained model."""
    checkpoint_path = config['checkpoint_path']
    print(f"\n{'='*60}")
    print(f"🧠 Running Inference for {config['num_layers']}-Layer Model")
    print(f"{'='*60}")

    # Load checkpoint
    with open(checkpoint_path, 'rb') as f:
        model_state = pickle.load(f)

    # Initialize model and load state
    rngs = nnx.Rngs(config['seed'])
    model = ViTForDepth_String2D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load and preprocess image
    img = cv2.cvtColor(cv2.imread(sample_image_path), cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0

    # Center crop to 224x224
    h, w, _ = img.shape
    y = max(0, (h - 224) // 2)
    x = max(0, (w - 224) // 2)
    img = img[y:y+224, x:x+224]

    img_batch = jnp.expand_dims(img, 0)

    # Run inference
    @nnx.jit
    def predict(model, img):
        return model(img, train=False)

    predicted_depth = predict(model, img_batch).squeeze()

    # Visualize results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(
        f"Depth Prediction using {config['num_layers']}-Layer ViT",
        fontsize=16,
        fontweight='bold'
    )

    ax1.imshow(img)
    ax1.set_title("Input RGB", fontsize=12)
    ax1.axis('off')

    im2 = ax2.imshow(predicted_depth, cmap='magma')
    ax2.set_title("Predicted Depth", fontsize=12)
    ax2.axis('off')

    # Add colorbar
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Save output
    os.makedirs("outputs", exist_ok=True)
    output_name = os.path.join(
        "outputs",
        f"prediction_{config['num_layers']}_layers.png"
    )
    plt.tight_layout()
    plt.savefig(output_name, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"🖼️  Inference plot saved to: {output_name}")
    print(f"   Depth range: [{float(predicted_depth.min()):.3f}, {float(predicted_depth.max()):.3f}]")


# ============================================================================
# SECTION 5: MAIN EXECUTION
# ============================================================================

def plot_training_comparison(all_results: Dict[str, List[float]]):
    """Plot and save training loss comparison."""
    plt.figure(figsize=(14, 8))

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

    for (name, losses), color in zip(all_results.items(), colors):
        # Smooth the curve for better visualization
        window_size = min(50, len(losses) // 10)
        if window_size > 1:
            smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
            x_smooth = np.arange(window_size//2, len(losses) - window_size//2 + 1)
            plt.plot(x_smooth, smoothed, label=name, linewidth=2.5, color=color, alpha=0.8)
        else:
            plt.plot(losses, label=name, linewidth=2.5, color=color, alpha=0.8)

    plt.title("Training Loss Comparison by Number of Layers", fontsize=18, fontweight='bold')
    plt.xlabel("Training Step", fontsize=14)
    plt.ylabel("Combined Loss (L1 + Gradient)", fontsize=14)
    plt.legend(loc='best', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, len(next(iter(all_results.values()))))

    # Add annotations for better readability
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    comparison_path = os.path.join("outputs", "loss_curve_comparison.png")
    plt.tight_layout()
    plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"📉 Final comparison plot saved to: {comparison_path}")


def main():
    """Main execution function."""
    print("\n" + "="*60)
    print(" " * 15 + "STRING2D DEPTH ESTIMATION")
    print(" " * 12 + "Corrected Flax NNX Implementation")
    print("="*60)

    # Define experiments to run
    layer_configs_to_test = [2, 4, 6, 12]
    all_results = {}

    # Run experiments
    for num_layers in layer_configs_to_test:
        config = {
            'seed': 42,
            'data_dir': "./data",
            'batch_size': 8,
            'steps': 5000,
            'lr': 3e-4,
            'num_layers': num_layers,
            'embed_dim': 192,
            'num_heads': 3,
            'checkpoint_path': f"./checkpoints/string2d_nnx_{num_layers}_layers_corrected.pkl"
        }

        # Train model
        final_checkpoint, losses = train_model(config)
        all_results[f'{num_layers} Layers'] = losses

        # Run inference if training successful
        if final_checkpoint:
            run_inference(
                config,
                sample_image_path="./data/input_frames/frame_0014.png"
            )

    # Plot comparison
    plot_training_comparison(all_results)

    print("\n" + "="*60)
    print(" " * 18 + "✨ ALL EXPERIMENTS COMPLETE!")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

Overwriting /content/vision_transformer_depth/nnx_string3d_experiment_pipeline.py


In [ ]:
# @title ### anyRES_nnx_string3d_experiment_pipeline
%%writefile /content/vision_transformer_depth/anyRES_nnx_string3d_experiment_pipeline.py
"""
Variable-Size Vision Transformer with String2D Position Encoding for Depth Estimation
Supports arbitrary input resolutions at inference time!
"""

import os
import sys
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
from typing import Optional, Tuple, Dict, List

import jax
import jax.numpy as jnp
from jax import random, jit

import flax.nnx as nnx
import optax
import grain
from einops import rearrange
from tqdm import trange


# ============================================================================
# SECTION 1: DATA LOADING (same as before)
# ============================================================================

class DepthDataSource:
    """Data source for loading RGB images and corresponding depth maps."""

    def __init__(self, data_dir: str = "."):
        """Initialize data source with directory containing input frames and depth maps."""
        self.pairs = [
            (f, os.path.join(data_dir, "depth_maps", os.path.basename(f)))
            for f in glob.glob(os.path.join(data_dir, "input_frames", "*.png"))
            if os.path.exists(os.path.join(data_dir, "depth_maps", os.path.basename(f)))
        ]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, jnp.ndarray]:
        """Load and preprocess image-depth pair at given index."""
        inp_path, dep_path = self.pairs[idx]

        # Load images
        inp_rgb = cv2.cvtColor(cv2.imread(inp_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(dep_path, cv2.IMREAD_GRAYSCALE)

        # Normalize to [0, 1]
        inp_rgb = inp_rgb.astype(np.float32) / 255.0
        dep = dep.astype(np.float32) / 255.0

        # Center crop to 224x224 for training
        h, w, _ = inp_rgb.shape
        y = max(0, (h - 224) // 2)
        x = max(0, (w - 224) // 2)

        inp_cropped = inp_rgb[y:y+224, x:x+224]
        dep_cropped = dep[y:y+224, x:x+224]

        return {
            'image': inp_cropped,
            'depth': dep_cropped[..., None]
        }


def create_data_iterator(data_dir: str, batch_size: int, seed: int = 42):
    """Create a data iterator using Grain for efficient data loading."""
    source = DepthDataSource(data_dir=data_dir)

    if not source.pairs:
        return iter([])

    dataset = (
        grain.MapDataset.source(source)
        .shuffle(seed=seed)
        .batch(batch_size, drop_remainder=True)
        .repeat()
        .to_iter_dataset()
    )

    return iter(dataset)


# ============================================================================
# SECTION 2: VARIABLE-SIZE MODEL ARCHITECTURE
# ============================================================================

class VariableStringPositionEmbedding2D(nnx.Module):
    """2D string-based position encoding that handles variable grid sizes."""

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize string position embedding module."""
        self.embed_dim = embed_dim

        # Cayley parameterization for orthogonal matrix
        self.S_cayley = nnx.Param(
            nnx.initializers.normal(stddev=0.01)(
                rngs.params(), (embed_dim, embed_dim)
            )
        )

    def __call__(
        self,
        x: jnp.ndarray,
        h_patches: int,
        w_patches: int
    ) -> jnp.ndarray:
        """Apply 2D string position encoding to input with variable grid size."""
        # Construct orthogonal matrix via Cayley transform
        S_antisym = (self.S_cayley.value - self.S_cayley.value.T) / 2.0
        I = jnp.eye(self.embed_dim, dtype=x.dtype)
        P = jnp.linalg.solve(I + S_antisym, I - S_antisym)

        # Apply orthogonal transformation
        x_transformed = jnp.matmul(x, P.T)

        # Generate spatial coordinates for variable grid
        y_grid, x_grid = jnp.meshgrid(
            jnp.arange(h_patches),
            jnp.arange(w_patches),
            indexing='ij'
        )

        # Include CLS token position (0, 0)
        x_coords = jnp.concatenate([jnp.zeros(1), x_grid.flatten() + 1])
        y_coords = jnp.concatenate([jnp.zeros(1), y_grid.flatten() + 1])

        # Normalize coordinates to [-1, 1]
        x_coords = (x_coords / max(w_patches, 1)) * 2 - 1
        y_coords = (y_coords / max(h_patches, 1)) * 2 - 1

        # Generate frequency encodings
        freqs = 1.0 / (10000 ** (
            jnp.arange(0, self.embed_dim // 2, dtype=jnp.float32) * 2 / self.embed_dim
        ))

        angles = (
            jnp.einsum('i,j->ij', x_coords, freqs) +
            jnp.einsum('i,j->ij', y_coords, freqs)
        )

        cos_vals = jnp.repeat(jnp.cos(angles), 2, axis=-1)
        sin_vals = jnp.repeat(jnp.sin(angles), 2, axis=-1)

        # Apply rotational encoding
        x1, x2 = jnp.split(x_transformed, 2, axis=-1)
        encoded = (
            x_transformed * cos_vals[None, None, :, :] +
            jnp.concatenate([-x2, x1], axis=-1) * sin_vals[None, None, :, :]
        )

        return encoded


class VariableString2DViTAttention(nnx.Module):
    """Multi-head attention with variable-size 2D string position encoding."""

    def __init__(
        self,
        num_heads: int,
        embed_dim: int,
        rngs: nnx.Rngs
    ):
        """Initialize attention module."""
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads

        # Attention layers
        self.qkv = nnx.Linear(embed_dim, embed_dim * 3, use_bias=False, rngs=rngs)
        self.out_proj = nnx.Linear(embed_dim, embed_dim, rngs=rngs)

        # Variable-size string position encoder
        self.string_encoder = VariableStringPositionEmbedding2D(
            embed_dim=self.head_dim,
            rngs=rngs
        )

    def __call__(self, x: jnp.ndarray, h_patches: int, w_patches: int) -> jnp.ndarray:
        """Apply attention with variable-size 2D position encoding."""
        b, s, e = x.shape

        # Generate QKV
        qkv = self.qkv(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape for multi-head attention
        q = q.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        # Apply variable-size string encoding
        q_encoded = self.string_encoder(q, h_patches, w_patches)
        k_encoded = self.string_encoder(k, h_patches, w_patches)

        # Compute attention scores
        scores = (q_encoded @ k_encoded.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)
        attn_weights = nnx.softmax(scores, axis=-1)

        # Apply attention and reshape
        output = attn_weights @ v
        output = output.transpose(0, 2, 1, 3).reshape(b, s, e)

        return self.out_proj(output)


class VariableViTTransformerBlock(nnx.Module):
    """Transformer block with variable-size string-based 2D attention."""

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        rngs: nnx.Rngs
    ):
        """Initialize transformer block."""
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Layers
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.attention = VariableString2DViTAttention(
            num_heads=num_heads,
            embed_dim=embed_dim,
            rngs=rngs
        )
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.mlp_fc1 = nnx.Linear(embed_dim, embed_dim * 4, rngs=rngs)
        self.mlp_fc2 = nnx.Linear(embed_dim * 4, embed_dim, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

    def __call__(
        self,
        x: jnp.ndarray,
        h_patches: int,
        w_patches: int,
        train: bool = True
    ) -> jnp.ndarray:
        """Forward pass through transformer block."""
        # Attention with residual
        attn_out = self.attention(self.norm1(x), h_patches, w_patches)
        attn_out = self.dropout(attn_out, deterministic=not train)
        x = x + attn_out

        # MLP with residual
        mlp_out = self.norm2(x)
        mlp_out = self.mlp_fc1(mlp_out)
        mlp_out = nnx.gelu(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        mlp_out = self.mlp_fc2(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        x = x + mlp_out

        return x


class AdaptiveDepthHead(nnx.Module):
    """Adaptive decoder head that upsamples to any target resolution."""

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize adaptive depth decoder head."""
        self.embed_dim = embed_dim

        # Progressive refinement layers
        self.proj = nnx.Linear(embed_dim, 256, rngs=rngs)

        self.refine1 = nnx.Conv(256, 128, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine2 = nnx.Conv(128, 64, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine3 = nnx.Conv(64, 32, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine4 = nnx.Conv(32, 16, kernel_size=(3, 3), padding=1, rngs=rngs)

        # Final convolution to single channel depth
        self.final_conv = nnx.Conv(16, 1, kernel_size=(1, 1), rngs=rngs)

    def __call__(
        self,
        patch_features: jnp.ndarray,
        h_patches: int,
        w_patches: int,
        target_h: int,
        target_w: int
    ) -> jnp.ndarray:
        """Decode patch features to depth map at target resolution."""
        # Project features
        x = self.proj(patch_features)

        # Reshape patches to spatial grid
        b = patch_features.shape[0]
        x = x.reshape(b, h_patches, w_patches, 256)

        # Progressive upsampling with dynamic sizing
        current_h, current_w = h_patches, w_patches

        # Stage 1: Upsample and refine
        scale = min(4, max(target_h // current_h, target_w // current_w))
        if scale > 1:
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 256), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine1(x))

        # Stage 2: Continue upsampling
        if current_h < target_h or current_w < target_w:
            scale = min(2, max(target_h // current_h, target_w // current_w))
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 128), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine2(x))

        # Stage 3
        if current_h < target_h or current_w < target_w:
            scale = min(2, max(target_h // current_h, target_w // current_w))
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 64), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine3(x))

        # Stage 4
        if current_h < target_h or current_w < target_w:
            x = jax.image.resize(x, (b, target_h, target_w, 32), method='bilinear')
        x = nnx.relu(self.refine4(x))

        # Final resize if needed (should already be at target)
        if x.shape[1:3] != (target_h, target_w):
            x = jax.image.resize(x, (b, target_h, target_w, 16), method='bilinear')

        # Final depth prediction
        x = self.final_conv(x)
        x = nnx.sigmoid(x)

        return x


class VariableSizeViTForDepth(nnx.Module):
    """Vision Transformer that handles arbitrary input sizes for depth estimation."""

    def __init__(
        self,
        patch_size: int,
        num_layers: int,
        embed_dim: int,
        num_heads: int,
        default_size: int,  # Size model is trained on (e.g., 224)
        rngs: nnx.Rngs
    ):
        """Initialize variable-size ViT model."""
        self.patch_size = patch_size
        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.default_size = default_size
        self.default_patches = default_size // patch_size  # e.g., 14 for 224/16

        # Patch embedding
        self.patch_embed = nnx.Conv(
            in_features=3,
            out_features=embed_dim,
            kernel_size=(patch_size, patch_size),
            strides=(patch_size, patch_size),
            rngs=rngs
        )

        # CLS token (learnable)
        self.cls_token = nnx.Param(
            nnx.initializers.normal(stddev=0.02)(
                rngs.params(), (1, 1, embed_dim)
            )
        )

        # Position embeddings for default size
        n_default_patches = self.default_patches * self.default_patches
        self.pos_embed = nnx.Param(
            nnx.initializers.normal(stddev=0.02)(
                rngs.params(), (1, n_default_patches + 1, embed_dim)  # +1 for CLS
            )
        )

        # Dropout
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

        # Transformer blocks
        self.blocks = [
            VariableViTTransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                rngs=rngs
            )
            for _ in range(num_layers)
        ]

        # Final norm and adaptive depth head
        self.norm = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.depth_head = AdaptiveDepthHead(embed_dim, rngs=rngs)

    def interpolate_pos_embeddings(
        self,
        h_patches: int,
        w_patches: int
    ) -> jnp.ndarray:
        """Interpolate position embeddings to match input size."""
        n_patches = h_patches * w_patches

        # If same as training size, return as is
        if h_patches == self.default_patches and w_patches == self.default_patches:
            return self.pos_embed.value

        # Split CLS and patch position embeddings
        cls_pos = self.pos_embed.value[:, :1, :]  # CLS token position
        patch_pos = self.pos_embed.value[:, 1:, :]  # Patch positions

        # Reshape patch positions to 2D grid
        patch_pos = patch_pos.reshape(
            1, self.default_patches, self.default_patches, self.embed_dim
        )

        # Interpolate to new size
        patch_pos = jax.image.resize(
            patch_pos,
            shape=(1, h_patches, w_patches, self.embed_dim),
            method='bilinear'
        )

        # Flatten back
        patch_pos = patch_pos.reshape(1, n_patches, self.embed_dim)

        # Concatenate CLS and interpolated patch positions
        interpolated = jnp.concatenate([cls_pos, patch_pos], axis=1)

        return interpolated

    def __call__(self, image: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        """Forward pass through the model - handles any input size divisible by patch_size."""
        b, h, w, c = image.shape

        # Check if dimensions are divisible by patch_size
        if h % self.patch_size != 0 or w % self.patch_size != 0:
            # Pad to nearest multiple of patch_size
            pad_h = (self.patch_size - h % self.patch_size) % self.patch_size
            pad_w = (self.patch_size - w % self.patch_size) % self.patch_size
            image = jnp.pad(
                image,
                ((0, 0), (0, pad_h), (0, pad_w), (0, 0)),
                mode='reflect'
            )
            padded_h = h + pad_h
            padded_w = w + pad_w
        else:
            padded_h = h
            padded_w = w

        # Extract patches
        patches = self.patch_embed(image)
        h_patches, w_patches = patches.shape[1], patches.shape[2]
        n_patches = h_patches * w_patches
        patches = patches.reshape(b, n_patches, self.embed_dim)

        # Add CLS token
        cls_tokens = jnp.tile(self.cls_token.value, (b, 1, 1))
        x = jnp.concatenate([cls_tokens, patches], axis=1)

        # Get interpolated position embeddings for current size
        pos_embed = self.interpolate_pos_embeddings(h_patches, w_patches)
        x = x + pos_embed

        # Apply dropout
        x = self.dropout(x, deterministic=not train)

        # Process through transformer blocks
        for block in self.blocks:
            x = block(x, h_patches, w_patches, train=train)

        # Apply final norm
        x = self.norm(x)

        # Remove CLS token and decode depth
        patch_features = x[:, 1:, :]  # Remove CLS token

        # Decode to original input resolution (before padding)
        depth_output = self.depth_head(
            patch_features,
            h_patches,
            w_patches,
            target_h=h,  # Use original height
            target_w=w   # Use original width
        )

        return depth_output


# ============================================================================
# SECTION 3: TRAINING (with fixed size)
# ============================================================================

def compute_loss(pred_depth: jnp.ndarray, gt_depth: jnp.ndarray) -> jnp.ndarray:
    """Compute depth estimation loss."""
    # L1 loss
    l1_loss = jnp.mean(jnp.abs(pred_depth - gt_depth))

    # Gradient loss for edge preservation
    def gradient(x):
        grad_x = x[:, 1:, :, :] - x[:, :-1, :, :]
        grad_y = x[:, :, 1:, :] - x[:, :, :-1, :]
        return grad_x, grad_y

    pred_grad_x, pred_grad_y = gradient(pred_depth)
    gt_grad_x, gt_grad_y = gradient(gt_depth)

    grad_loss = (
        jnp.mean(jnp.abs(pred_grad_x - gt_grad_x)) +
        jnp.mean(jnp.abs(pred_grad_y - gt_grad_y))
    )

    # Combined loss
    total_loss = l1_loss + 0.5 * grad_loss

    return total_loss


def train_model(config: Dict) -> Tuple[Optional[str], List[float]]:
    """Train the variable-size depth estimation model."""
    print(f"\n{'='*60}")
    print(f"🚀 Starting Training: Variable-Size {config['num_layers']}-Layer Model")
    print(f"   Training on {config['train_size']}×{config['train_size']}")
    print(f"   Model supports inference at ANY resolution!")
    print(f"{'='*60}")

    # Initialize RNGs
    rngs = nnx.Rngs(config['seed'])

    # Create data iterator
    data_iter = create_data_iterator(
        config['data_dir'],
        config['batch_size'],
        config['seed']
    )

    # Get sample batch
    try:
        batch = next(data_iter)
    except StopIteration:
        print("Error: Data iterator is empty.")
        return None, []

    # Initialize model
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],  # Training size
        rngs=rngs
    )

    # Initialize optimizer
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=config['lr'],
        warmup_steps=500,
        decay_steps=config['steps'],
        end_value=1e-6
    )

    optimizer = nnx.Optimizer(
        model,
        optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adam(schedule)
        )
    )

    @nnx.jit
    def train_step(model, optimizer, batch):
        """Single training step."""
        images = batch['image']
        gt_depths = batch['depth']

        def loss_fn(model):
            pred_depths = model(images, train=True)
            return compute_loss(pred_depths, gt_depths)

        loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(grads)

        return loss

    # Training loop
    all_losses = []
    pbar = trange(
        config['steps'],
        desc=f"🔥 Training Variable-Size Model ({config['num_layers']} Layers)"
    )

    for step in pbar:
        batch = next(data_iter)
        loss = train_step(model, optimizer, batch)
        loss_val = float(loss)
        all_losses.append(loss_val)

        pbar.set_postfix(loss=f"{loss_val:.4f}")

        if step % 100 == 0 and step > 0:
            avg_loss = np.mean(all_losses[-100:])
            print(f"\n[Step {step}] Average Loss: {avg_loss:.4f}")

    # Save checkpoint
    checkpoint_dir = os.path.dirname(config['checkpoint_path'])
    os.makedirs(checkpoint_dir, exist_ok=True)

    model_state = nnx.state(model)
    with open(config['checkpoint_path'], "wb") as f:
        pickle.dump(model_state, f)

    print(f"✅ Training complete! Model saved to {config['checkpoint_path']}")
    print(f"   Final loss: {all_losses[-1]:.4f}")

    return config['checkpoint_path'], all_losses


# ============================================================================
# SECTION 4: MULTI-RESOLUTION INFERENCE
# ============================================================================

def run_multires_inference(config: Dict, test_image_path: str):
    """Test model on multiple resolutions to demonstrate variable-size capability."""
    print(f"\n{'='*60}")
    print(f"🎯 Multi-Resolution Inference Test")
    print(f"   Model trained on: {config['train_size']}×{config['train_size']}")
    print(f"{'='*60}")

    # Load checkpoint
    with open(config['checkpoint_path'], 'rb') as f:
        model_state = pickle.load(f)

    # Initialize model
    rngs = nnx.Rngs(config['seed'])
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load test image
    img_original = cv2.cvtColor(cv2.imread(test_image_path), cv2.COLOR_BGR2RGB)
    img_original = img_original.astype(np.float32) / 255.0

    # Test at multiple resolutions
    test_sizes = [
        (224, 224),   # Training size
        (384, 384),   # 1.7x larger
        (512, 512),   # 2.3x larger
        (320, 480),   # Different aspect ratio
    ]

    @nnx.jit
    def predict(model, img):
        return model(img, train=False)

    # Create visualization
    fig = plt.figure(figsize=(20, 12))

    for idx, (h, w) in enumerate(test_sizes):
        # Resize image
        img_resized = cv2.resize(img_original, (w, h))
        img_batch = jnp.expand_dims(img_resized, 0)

        # Run inference
        depth_pred = predict(model, img_batch).squeeze()

        # Plot RGB
        ax1 = plt.subplot(len(test_sizes), 2, idx * 2 + 1)
        ax1.imshow(img_resized)
        ax1.set_title(f"Input RGB ({h}×{w})", fontsize=12, fontweight='bold')
        ax1.axis('off')

        # Plot Depth
        ax2 = plt.subplot(len(test_sizes), 2, idx * 2 + 2)
        im = ax2.imshow(depth_pred, cmap='magma')
        ax2.set_title(f"Predicted Depth ({h}×{w})", fontsize=12, fontweight='bold')
        ax2.axis('off')
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

        print(f"✓ Inference at {h}×{w}: Depth range [{float(depth_pred.min()):.3f}, {float(depth_pred.max()):.3f}]")

    plt.suptitle(
        f"Variable-Size ViT: Trained on {config['train_size']}×{config['train_size']}, Tested on Multiple Resolutions",
        fontsize=16,
        fontweight='bold'
    )
    plt.tight_layout()

    # Save output
    os.makedirs("outputs", exist_ok=True)
    output_path = os.path.join("outputs", f"multires_inference_{config['num_layers']}_layers.png")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Multi-resolution results saved to: {output_path}")


def test_extreme_resolutions(config: Dict, test_image_path: str):
    """Test model on extreme resolutions to show robustness."""
    print(f"\n{'='*60}")
    print(f"🔬 Extreme Resolution Test")
    print(f"{'='*60}")

    # Load model
    with open(config['checkpoint_path'], 'rb') as f:
        model_state = pickle.load(f)

    rngs = nnx.Rngs(config['seed'])
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load test image
    img_original = cv2.cvtColor(cv2.imread(test_image_path), cv2.COLOR_BGR2RGB)
    img_original = img_original.astype(np.float32) / 255.0

    # Test extreme cases
    extreme_sizes = [
        (160, 160),   # Smaller than training
        (768, 768),   # Much larger
        (256, 512),   # 1:2 aspect ratio
        (640, 320),   # 2:1 aspect ratio
    ]

    for h, w in extreme_sizes:
        try:
            img_resized = cv2.resize(img_original, (w, h))
            img_batch = jnp.expand_dims(img_resized, 0)

            # Time the inference
            import time
            start = time.time()
            depth_pred = model(img_batch, train=False)
            elapsed = time.time() - start

            print(f"✓ {h}×{w} ({h*w/1000:.1f}K pixels): {elapsed:.3f}s, "
                  f"Output shape: {depth_pred.shape}")

        except Exception as e:
            print(f"✗ {h}×{w} failed: {str(e)}")


# ============================================================================
# SECTION 5: MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function."""
    print("\n" + "="*60)
    print(" " * 10 + "VARIABLE-SIZE VISION TRANSFORMER")
    print(" " * 15 + "FOR DEPTH ESTIMATION")
    print(" " * 8 + "Train Once, Infer at Any Resolution!")
    print("="*60)

    # Configuration
    config = {
        'seed': 42,
        'data_dir': "./data",
        'batch_size': 8,
        'steps': 3000,  # Reduced for demo
        'lr': 3e-4,
        'num_layers': 6,
        'embed_dim': 192,
        'num_heads': 3,
        'train_size': 224,  # Training resolution
        'checkpoint_path': "./checkpoints/variable_size_vit_6_layers.pkl"
    }

    # Train model (on fixed 224×224)
    final_checkpoint, losses = train_model(config)

    if final_checkpoint:
        # Test on multiple resolutions
        test_image = "./data/input_frames/frame_0014.png"

        # Run multi-resolution inference
        run_multires_inference(config, test_image)

        # Test extreme resolutions
        test_extreme_resolutions(config, test_image)

    # Plot training curve
    if losses:
        plt.figure(figsize=(10, 6))
        plt.plot(losses, linewidth=2, color='#45B7D1', alpha=0.8)
        plt.title("Variable-Size ViT Training Loss", fontsize=16, fontweight='bold')
        plt.xlabel("Training Step", fontsize=12)
        plt.ylabel("Combined Loss", fontsize=12)
        plt.grid(True, alpha=0.3)

        os.makedirs("outputs", exist_ok=True)
        plt.savefig("outputs/variable_vit_training_loss.png", dpi=150, bbox_inches='tight')
        plt.show()

    print("\n" + "="*60)
    print(" " * 15 + "✨ EXPERIMENT COMPLETE!")
    print(" " * 8 + "Model works at ANY resolution!")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

In [ ]:
# @title ### anyRES_nnx_string3d_experiment_pipeline with input, target, prediction plotting
%%writefile /content/vision_transformer_depth/anyRES_nnx_string3d_experiment_pipeline.py

"""
Variable-Size Vision Transformer with String2D Position Encoding for Depth Estimation
Supports arbitrary input resolutions at inference time!
"""

import os
import sys
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
from typing import Optional, Tuple, Dict, List

import jax
import jax.numpy as jnp
from jax import random, jit

import flax.nnx as nnx
import optax
import grain
from einops import rearrange
from tqdm import trange


# ============================================================================
# SECTION 1: DATA LOADING (same as before)
# ============================================================================

class DepthDataSource:
    """Data source for loading RGB images and corresponding depth maps."""

    def __init__(self, data_dir: str = "."):
        """Initialize data source with directory containing input frames and depth maps."""
        self.pairs = [
            (f, os.path.join(data_dir, "depth_maps", os.path.basename(f)))
            for f in glob.glob(os.path.join(data_dir, "input_frames", "*.png"))
            if os.path.exists(os.path.join(data_dir, "depth_maps", os.path.basename(f)))
        ]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, jnp.ndarray]:
        """Load and preprocess image-depth pair at given index."""
        inp_path, dep_path = self.pairs[idx]

        # Load images
        inp_rgb = cv2.cvtColor(cv2.imread(inp_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(dep_path, cv2.IMREAD_GRAYSCALE)

        # Normalize to [0, 1]
        inp_rgb = inp_rgb.astype(np.float32) / 255.0
        dep = dep.astype(np.float32) / 255.0

        # Center crop to 224x224 for training
        h, w, _ = inp_rgb.shape
        y = max(0, (h - 224) // 2)
        x = max(0, (w - 224) // 2)

        inp_cropped = inp_rgb[y:y+224, x:x+224]
        dep_cropped = dep[y:y+224, x:x+224]

        return {
            'image': inp_cropped,
            'depth': dep_cropped[..., None]
        }


def create_data_iterator(data_dir: str, batch_size: int, seed: int = 42):
    """Create a data iterator using Grain for efficient data loading."""
    source = DepthDataSource(data_dir=data_dir)

    if not source.pairs:
        return iter([])

    dataset = (
        grain.MapDataset.source(source)
        .shuffle(seed=seed)
        .batch(batch_size, drop_remainder=True)
        .repeat()
        .to_iter_dataset()
    )

    return iter(dataset)


# ============================================================================
# SECTION 2: VARIABLE-SIZE MODEL ARCHITECTURE
# ============================================================================

class VariableStringPositionEmbedding2D(nnx.Module):
    """2D string-based position encoding that handles variable grid sizes."""

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize string position embedding module."""
        self.embed_dim = embed_dim

        # Cayley parameterization for orthogonal matrix
        self.S_cayley = nnx.Param(
            nnx.initializers.normal(stddev=0.01)(
                rngs.params(), (embed_dim, embed_dim)
            )
        )

    def __call__(
        self,
        x: jnp.ndarray,
        h_patches: int,
        w_patches: int
    ) -> jnp.ndarray:
        """Apply 2D string position encoding to input with variable grid size."""
        # Construct orthogonal matrix via Cayley transform
        S_antisym = (self.S_cayley.value - self.S_cayley.value.T) / 2.0
        I = jnp.eye(self.embed_dim, dtype=x.dtype)
        P = jnp.linalg.solve(I + S_antisym, I - S_antisym)

        # Apply orthogonal transformation
        x_transformed = jnp.matmul(x, P.T)

        # Generate spatial coordinates for variable grid
        y_grid, x_grid = jnp.meshgrid(
            jnp.arange(h_patches),
            jnp.arange(w_patches),
            indexing='ij'
        )

        # Include CLS token position (0, 0)
        x_coords = jnp.concatenate([jnp.zeros(1), x_grid.flatten() + 1])
        y_coords = jnp.concatenate([jnp.zeros(1), y_grid.flatten() + 1])

        # Normalize coordinates to [-1, 1]
        x_coords = (x_coords / max(w_patches, 1)) * 2 - 1
        y_coords = (y_coords / max(h_patches, 1)) * 2 - 1

        # Generate frequency encodings
        freqs = 1.0 / (10000 ** (
            jnp.arange(0, self.embed_dim // 2, dtype=jnp.float32) * 2 / self.embed_dim
        ))

        angles = (
            jnp.einsum('i,j->ij', x_coords, freqs) +
            jnp.einsum('i,j->ij', y_coords, freqs)
        )

        cos_vals = jnp.repeat(jnp.cos(angles), 2, axis=-1)
        sin_vals = jnp.repeat(jnp.sin(angles), 2, axis=-1)

        # Apply rotational encoding
        x1, x2 = jnp.split(x_transformed, 2, axis=-1)
        encoded = (
            x_transformed * cos_vals[None, None, :, :] +
            jnp.concatenate([-x2, x1], axis=-1) * sin_vals[None, None, :, :]
        )

        return encoded


class VariableString2DViTAttention(nnx.Module):
    """Multi-head attention with variable-size 2D string position encoding."""

    def __init__(
        self,
        num_heads: int,
        embed_dim: int,
        rngs: nnx.Rngs
    ):
        """Initialize attention module."""
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads

        # Attention layers
        self.qkv = nnx.Linear(embed_dim, embed_dim * 3, use_bias=False, rngs=rngs)
        self.out_proj = nnx.Linear(embed_dim, embed_dim, rngs=rngs)

        # Variable-size string position encoder
        self.string_encoder = VariableStringPositionEmbedding2D(
            embed_dim=self.head_dim,
            rngs=rngs
        )

    def __call__(self, x: jnp.ndarray, h_patches: int, w_patches: int) -> jnp.ndarray:
        """Apply attention with variable-size 2D position encoding."""
        b, s, e = x.shape

        # Generate QKV
        qkv = self.qkv(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape for multi-head attention
        q = q.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(b, s, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        # Apply variable-size string encoding
        q_encoded = self.string_encoder(q, h_patches, w_patches)
        k_encoded = self.string_encoder(k, h_patches, w_patches)

        # Compute attention scores
        scores = (q_encoded @ k_encoded.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)
        attn_weights = nnx.softmax(scores, axis=-1)

        # Apply attention and reshape
        output = attn_weights @ v
        output = output.transpose(0, 2, 1, 3).reshape(b, s, e)

        return self.out_proj(output)


class VariableViTTransformerBlock(nnx.Module):
    """Transformer block with variable-size string-based 2D attention."""

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        rngs: nnx.Rngs
    ):
        """Initialize transformer block."""
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Layers
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.attention = VariableString2DViTAttention(
            num_heads=num_heads,
            embed_dim=embed_dim,
            rngs=rngs
        )
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.mlp_fc1 = nnx.Linear(embed_dim, embed_dim * 4, rngs=rngs)
        self.mlp_fc2 = nnx.Linear(embed_dim * 4, embed_dim, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

    def __call__(
        self,
        x: jnp.ndarray,
        h_patches: int,
        w_patches: int,
        train: bool = True
    ) -> jnp.ndarray:
        """Forward pass through transformer block."""
        # Attention with residual
        attn_out = self.attention(self.norm1(x), h_patches, w_patches)
        attn_out = self.dropout(attn_out, deterministic=not train)
        x = x + attn_out

        # MLP with residual
        mlp_out = self.norm2(x)
        mlp_out = self.mlp_fc1(mlp_out)
        mlp_out = nnx.gelu(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        mlp_out = self.mlp_fc2(mlp_out)
        mlp_out = self.dropout(mlp_out, deterministic=not train)
        x = x + mlp_out

        return x


class AdaptiveDepthHead(nnx.Module):
    """Adaptive decoder head that upsamples to any target resolution."""

    def __init__(self, embed_dim: int, rngs: nnx.Rngs):
        """Initialize adaptive depth decoder head."""
        self.embed_dim = embed_dim

        # Progressive refinement layers
        self.proj = nnx.Linear(embed_dim, 256, rngs=rngs)

        self.refine1 = nnx.Conv(256, 128, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine2 = nnx.Conv(128, 64, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine3 = nnx.Conv(64, 32, kernel_size=(3, 3), padding=1, rngs=rngs)
        self.refine4 = nnx.Conv(32, 16, kernel_size=(3, 3), padding=1, rngs=rngs)

        # Final convolution to single channel depth
        self.final_conv = nnx.Conv(16, 1, kernel_size=(1, 1), rngs=rngs)

    def __call__(
        self,
        patch_features: jnp.ndarray,
        h_patches: int,
        w_patches: int,
        target_h: int,
        target_w: int
    ) -> jnp.ndarray:
        """Decode patch features to depth map at target resolution."""
        # Project features
        x = self.proj(patch_features)

        # Reshape patches to spatial grid
        b = patch_features.shape[0]
        x = x.reshape(b, h_patches, w_patches, 256)

        # Progressive upsampling with dynamic sizing
        current_h, current_w = h_patches, w_patches

        # Stage 1: Upsample and refine
        scale = min(4, max(target_h // current_h, target_w // current_w))
        if scale > 1:
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 256), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine1(x))

        # Stage 2: Continue upsampling
        if current_h < target_h or current_w < target_w:
            scale = min(2, max(target_h // current_h, target_w // current_w))
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 128), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine2(x))

        # Stage 3
        if current_h < target_h or current_w < target_w:
            scale = min(2, max(target_h // current_h, target_w // current_w))
            new_h = min(current_h * scale, target_h)
            new_w = min(current_w * scale, target_w)
            x = jax.image.resize(x, (b, new_h, new_w, 64), method='bilinear')
            current_h, current_w = new_h, new_w
        x = nnx.relu(self.refine3(x))

        # Stage 4
        if current_h < target_h or current_w < target_w:
            x = jax.image.resize(x, (b, target_h, target_w, 32), method='bilinear')
        x = nnx.relu(self.refine4(x))

        # Final resize if needed (should already be at target)
        if x.shape[1:3] != (target_h, target_w):
            x = jax.image.resize(x, (b, target_h, target_w, 16), method='bilinear')

        # Final depth prediction
        x = self.final_conv(x)
        x = nnx.sigmoid(x)

        return x


class VariableSizeViTForDepth(nnx.Module):
    """Vision Transformer that handles arbitrary input sizes for depth estimation."""

    def __init__(
        self,
        patch_size: int,
        num_layers: int,
        embed_dim: int,
        num_heads: int,
        default_size: int,  # Size model is trained on (e.g., 224)
        rngs: nnx.Rngs
    ):
        """Initialize variable-size ViT model."""
        self.patch_size = patch_size
        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.default_size = default_size
        self.default_patches = default_size // patch_size  # e.g., 14 for 224/16

        # Patch embedding
        self.patch_embed = nnx.Conv(
            in_features=3,
            out_features=embed_dim,
            kernel_size=(patch_size, patch_size),
            strides=(patch_size, patch_size),
            rngs=rngs
        )

        # CLS token (learnable)
        self.cls_token = nnx.Param(
            nnx.initializers.normal(stddev=0.02)(
                rngs.params(), (1, 1, embed_dim)
            )
        )

        # Position embeddings for default size
        n_default_patches = self.default_patches * self.default_patches
        self.pos_embed = nnx.Param(
            nnx.initializers.normal(stddev=0.02)(
                rngs.params(), (1, n_default_patches + 1, embed_dim)  # +1 for CLS
            )
        )

        # Dropout
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

        # Transformer blocks
        self.blocks = [
            VariableViTTransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                rngs=rngs
            )
            for _ in range(num_layers)
        ]

        # Final norm and adaptive depth head
        self.norm = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.depth_head = AdaptiveDepthHead(embed_dim, rngs=rngs)

    def interpolate_pos_embeddings(
        self,
        h_patches: int,
        w_patches: int
    ) -> jnp.ndarray:
        """Interpolate position embeddings to match input size."""
        n_patches = h_patches * w_patches

        # If same as training size, return as is
        if h_patches == self.default_patches and w_patches == self.default_patches:
            return self.pos_embed.value

        # Split CLS and patch position embeddings
        cls_pos = self.pos_embed.value[:, :1, :]  # CLS token position
        patch_pos = self.pos_embed.value[:, 1:, :]  # Patch positions

        # Reshape patch positions to 2D grid
        patch_pos = patch_pos.reshape(
            1, self.default_patches, self.default_patches, self.embed_dim
        )

        # Interpolate to new size
        patch_pos = jax.image.resize(
            patch_pos,
            shape=(1, h_patches, w_patches, self.embed_dim),
            method='bilinear'
        )

        # Flatten back
        patch_pos = patch_pos.reshape(1, n_patches, self.embed_dim)

        # Concatenate CLS and interpolated patch positions
        interpolated = jnp.concatenate([cls_pos, patch_pos], axis=1)

        return interpolated

    def __call__(self, image: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        """Forward pass through the model - handles any input size divisible by patch_size."""
        b, h, w, c = image.shape

        # Check if dimensions are divisible by patch_size
        if h % self.patch_size != 0 or w % self.patch_size != 0:
            # Pad to nearest multiple of patch_size
            pad_h = (self.patch_size - h % self.patch_size) % self.patch_size
            pad_w = (self.patch_size - w % self.patch_size) % self.patch_size
            image = jnp.pad(
                image,
                ((0, 0), (0, pad_h), (0, pad_w), (0, 0)),
                mode='reflect'
            )
            padded_h = h + pad_h
            padded_w = w + pad_w
        else:
            padded_h = h
            padded_w = w

        # Extract patches
        patches = self.patch_embed(image)
        h_patches, w_patches = patches.shape[1], patches.shape[2]
        n_patches = h_patches * w_patches
        patches = patches.reshape(b, n_patches, self.embed_dim)

        # Add CLS token
        cls_tokens = jnp.tile(self.cls_token.value, (b, 1, 1))
        x = jnp.concatenate([cls_tokens, patches], axis=1)

        # Get interpolated position embeddings for current size
        pos_embed = self.interpolate_pos_embeddings(h_patches, w_patches)
        x = x + pos_embed

        # Apply dropout
        x = self.dropout(x, deterministic=not train)

        # Process through transformer blocks
        for block in self.blocks:
            x = block(x, h_patches, w_patches, train=train)

        # Apply final norm
        x = self.norm(x)

        # Remove CLS token and decode depth
        patch_features = x[:, 1:, :]  # Remove CLS token

        # Decode to original input resolution (before padding)
        depth_output = self.depth_head(
            patch_features,
            h_patches,
            w_patches,
            target_h=h,  # Use original height
            target_w=w   # Use original width
        )

        return depth_output


# ============================================================================
# SECTION 3: TRAINING (with fixed size)
# ============================================================================

def compute_loss(pred_depth: jnp.ndarray, gt_depth: jnp.ndarray) -> jnp.ndarray:
    """Compute depth estimation loss."""
    # L1 loss
    l1_loss = jnp.mean(jnp.abs(pred_depth - gt_depth))

    # Gradient loss for edge preservation
    def gradient(x):
        grad_x = x[:, 1:, :, :] - x[:, :-1, :, :]
        grad_y = x[:, :, 1:, :] - x[:, :, :-1, :]
        return grad_x, grad_y

    pred_grad_x, pred_grad_y = gradient(pred_depth)
    gt_grad_x, gt_grad_y = gradient(gt_depth)

    grad_loss = (
        jnp.mean(jnp.abs(pred_grad_x - gt_grad_x)) +
        jnp.mean(jnp.abs(pred_grad_y - gt_grad_y))
    )

    # Combined loss
    total_loss = l1_loss + 0.5 * grad_loss

    return total_loss


def train_model(config: Dict) -> Tuple[Optional[str], List[float]]:
    """Train the variable-size depth estimation model."""
    print(f"\n{'='*60}")
    print(f"🚀 Starting Training: Variable-Size {config['num_layers']}-Layer Model")
    print(f"   Training on {config['train_size']}×{config['train_size']}")
    print(f"   Model supports inference at ANY resolution!")
    print(f"{'='*60}")

    # Initialize RNGs
    rngs = nnx.Rngs(config['seed'])

    # Create data iterator
    data_iter = create_data_iterator(
        config['data_dir'],
        config['batch_size'],
        config['seed']
    )

    # Get sample batch
    try:
        batch = next(data_iter)
    except StopIteration:
        print("Error: Data iterator is empty.")
        return None, []

    # Initialize model
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],  # Training size
        rngs=rngs
    )

    # Initialize optimizer
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=config['lr'],
        warmup_steps=500,
        decay_steps=config['steps'],
        end_value=1e-6
    )

    optimizer = nnx.Optimizer(
        model,
        optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adam(schedule)
        )
    )

    @nnx.jit
    def train_step(model, optimizer, batch):
        """Single training step."""
        images = batch['image']
        gt_depths = batch['depth']

        def loss_fn(model):
            pred_depths = model(images, train=True)
            return compute_loss(pred_depths, gt_depths)

        loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(grads)

        return loss

    # Training loop
    all_losses = []
    pbar = trange(
        config['steps'],
        desc=f"🔥 Training Variable-Size Model ({config['num_layers']} Layers)"
    )

    for step in pbar:
        batch = next(data_iter)
        loss = train_step(model, optimizer, batch)
        loss_val = float(loss)
        all_losses.append(loss_val)

        pbar.set_postfix(loss=f"{loss_val:.4f}")

        if step % 100 == 0 and step > 0:
            avg_loss = np.mean(all_losses[-100:])
            print(f"\n[Step {step}] Average Loss: {avg_loss:.4f}")

    # Save checkpoint
    checkpoint_dir = os.path.dirname(config['checkpoint_path'])
    os.makedirs(checkpoint_dir, exist_ok=True)

    model_state = nnx.state(model)
    with open(config['checkpoint_path'], "wb") as f:
        pickle.dump(model_state, f)

    print(f"✅ Training complete! Model saved to {config['checkpoint_path']}")
    print(f"   Final loss: {all_losses[-1]:.4f}")

    return config['checkpoint_path'], all_losses


# ============================================================================
# SECTION 4: MULTI-RESOLUTION INFERENCE WITH GROUND TRUTH
# ============================================================================

def run_single_inference_comparison(config: Dict, test_image_path: str):
    """Run inference on a single image with ground truth comparison."""
    print(f"\n{'='*60}")
    print(f"🔍 Single Image Inference with Ground Truth Comparison")
    print(f"{'='*60}")

    # Load checkpoint
    with open(config['checkpoint_path'], 'rb') as f:
        model_state = pickle.load(f)

    # Initialize model
    rngs = nnx.Rngs(config['seed'])
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load test image
    img_original = cv2.cvtColor(cv2.imread(test_image_path), cv2.COLOR_BGR2RGB)
    img_original = img_original.astype(np.float32) / 255.0

    # Load ground truth depth
    depth_path = test_image_path.replace("input_frames", "depth_maps")
    if os.path.exists(depth_path):
        depth_gt = cv2.imread(depth_path, cv2.IMREAD_GRAYSCALE)
        depth_gt = depth_gt.astype(np.float32) / 255.0
    else:
        print("⚠️ Ground truth depth not found")
        depth_gt = np.zeros_like(img_original[:, :, 0])

    # Crop to 224x224 for fair comparison
    h, w = 224, 224
    y_start = (img_original.shape[0] - h) // 2
    x_start = (img_original.shape[1] - w) // 2

    img_cropped = img_original[y_start:y_start+h, x_start:x_start+w]
    depth_gt_cropped = depth_gt[y_start:y_start+h, x_start:x_start+w]

    # Run inference
    img_batch = jnp.expand_dims(img_cropped, 0)

    @nnx.jit
    def predict(model, img):
        return model(img, train=False)

    depth_pred = predict(model, img_batch).squeeze()

    # Calculate comprehensive metrics
    mae = np.mean(np.abs(depth_pred - depth_gt_cropped))
    rmse = np.sqrt(np.mean((depth_pred - depth_gt_cropped)**2))

    # Relative metrics (avoiding division by zero)
    mask = depth_gt_cropped > 0.01
    if mask.any():
        rel = np.mean(np.abs(depth_pred[mask] - depth_gt_cropped[mask]) / depth_gt_cropped[mask])
    else:
        rel = 0

    # Create detailed visualization
    fig = plt.figure(figsize=(20, 8))

    # Input RGB
    ax1 = plt.subplot(2, 4, 1)
    ax1.imshow(img_cropped)
    ax1.set_title("Input RGB\n(224×224)", fontsize=12, fontweight='bold')
    ax1.axis('off')

    # Ground Truth Depth
    ax2 = plt.subplot(2, 4, 2)
    im2 = ax2.imshow(depth_gt_cropped, cmap='magma', vmin=0, vmax=1)
    ax2.set_title(f"Ground Truth Depth\nRange: [{depth_gt_cropped.min():.3f}, {depth_gt_cropped.max():.3f}]",
                  fontsize=12, fontweight='bold')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Predicted Depth
    ax3 = plt.subplot(2, 4, 3)
    im3 = ax3.imshow(depth_pred, cmap='magma', vmin=0, vmax=1)
    ax3.set_title(f"Predicted Depth\nRange: [{depth_pred.min():.3f}, {depth_pred.max():.3f}]",
                  fontsize=12, fontweight='bold')
    ax3.axis('off')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    # Error Map
    ax4 = plt.subplot(2, 4, 4)
    error_map = np.abs(depth_pred - depth_gt_cropped)
    im4 = ax4.imshow(error_map, cmap='hot', vmin=0, vmax=0.3)
    ax4.set_title(f"Absolute Error\nMean: {error_map.mean():.3f}",
                  fontsize=12, fontweight='bold')
    ax4.axis('off')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

    # Histogram comparison
    ax5 = plt.subplot(2, 4, 5)
    ax5.hist(depth_gt_cropped.flatten(), bins=50, alpha=0.5, label='Ground Truth', color='blue')
    ax5.hist(depth_pred.flatten(), bins=50, alpha=0.5, label='Predicted', color='red')
    ax5.set_xlabel('Depth Value')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Depth Distribution', fontsize=12, fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # Cross-section comparison (horizontal slice through middle)
    ax6 = plt.subplot(2, 4, 6)
    mid_row = h // 2
    ax6.plot(depth_gt_cropped[mid_row, :], label='Ground Truth', linewidth=2, color='blue')
    ax6.plot(depth_pred[mid_row, :], label='Predicted', linewidth=2, color='red', linestyle='--')
    ax6.set_xlabel('Pixel Position (x)')
    ax6.set_ylabel('Depth Value')
    ax6.set_title(f'Horizontal Cross-Section (y={mid_row})', fontsize=12, fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)

    # Scatter plot: GT vs Predicted
    ax7 = plt.subplot(2, 4, 7)
    sample_indices = np.random.choice(depth_gt_cropped.size, 5000, replace=False)
    gt_sample = depth_gt_cropped.flatten()[sample_indices]
    pred_sample = depth_pred.flatten()[sample_indices]
    ax7.scatter(gt_sample, pred_sample, alpha=0.3, s=1)
    ax7.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Prediction')
    ax7.set_xlabel('Ground Truth Depth')
    ax7.set_ylabel('Predicted Depth')
    ax7.set_title('Correlation Plot', fontsize=12, fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    ax7.set_xlim([0, 1])
    ax7.set_ylim([0, 1])

    # Metrics display
    ax8 = plt.subplot(2, 4, 8)
    ax8.axis('off')
    metrics_text = f"""
    Performance Metrics:

    MAE:  {mae:.4f}
    RMSE: {rmse:.4f}
    Rel:  {rel:.4f}

    Min Error: {error_map.min():.4f}
    Max Error: {error_map.max():.4f}
    Std Error: {error_map.std():.4f}

    Correlation: {np.corrcoef(depth_gt_cropped.flatten(), depth_pred.flatten())[0,1]:.4f}
    """
    ax8.text(0.1, 0.5, metrics_text, fontsize=12, family='monospace',
             verticalalignment='center', bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.5))

    plt.suptitle(
        f"{config['num_layers']}-Layer ViT Depth Estimation: Detailed Analysis",
        fontsize=14,
        fontweight='bold'
    )
    plt.tight_layout()

    # Save output
    os.makedirs("outputs", exist_ok=True)
    output_path = os.path.join("outputs", f"detailed_comparison_{config['num_layers']}_layers.png")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Detailed comparison saved to: {output_path}")
    print(f"\n📈 Final Metrics:")
    print(f"   MAE:  {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   Relative Error: {rel:.4f}")
    print(f"   Correlation: {np.corrcoef(depth_gt_cropped.flatten(), depth_pred.flatten())[0,1]:.4f}")

def run_multires_inference(config: Dict, test_image_path: str):
    """Test model on multiple resolutions with ground truth comparison."""
    print(f"\n{'='*60}")
    print(f"🎯 Multi-Resolution Inference Test with Ground Truth")
    print(f"   Model trained on: {config['train_size']}×{config['train_size']}")
    print(f"{'='*60}")

    # Load checkpoint
    with open(config['checkpoint_path'], 'rb') as f:
        model_state = pickle.load(f)

    # Initialize model
    rngs = nnx.Rngs(config['seed'])
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load test image and corresponding depth
    img_original = cv2.cvtColor(cv2.imread(test_image_path), cv2.COLOR_BGR2RGB)
    img_original = img_original.astype(np.float32) / 255.0

    # Load ground truth depth
    depth_path = test_image_path.replace("input_frames", "depth_maps")
    if os.path.exists(depth_path):
        depth_gt = cv2.imread(depth_path, cv2.IMREAD_GRAYSCALE)
        depth_gt = depth_gt.astype(np.float32) / 255.0
    else:
        print("⚠️ Ground truth depth not found, using placeholder")
        depth_gt = np.zeros_like(img_original[:, :, 0])

    # Test at multiple resolutions
    test_sizes = [
        (224, 224),   # Training size
        (384, 384),   # 1.7x larger
        (512, 512),   # 2.3x larger
        (320, 480),   # Different aspect ratio
    ]

    @nnx.jit
    def predict(model, img):
        return model(img, train=False)

    # Create visualization with ground truth
    fig = plt.figure(figsize=(24, 16))

    for idx, (h, w) in enumerate(test_sizes):
        # Resize image and ground truth
        img_resized = cv2.resize(img_original, (w, h))
        depth_gt_resized = cv2.resize(depth_gt, (w, h))
        img_batch = jnp.expand_dims(img_resized, 0)

        # Run inference
        depth_pred = predict(model, img_batch).squeeze()

        # Calculate error metrics
        mae = np.mean(np.abs(depth_pred - depth_gt_resized))
        rmse = np.sqrt(np.mean((depth_pred - depth_gt_resized)**2))

        # Plot RGB
        ax1 = plt.subplot(len(test_sizes), 4, idx * 4 + 1)
        ax1.imshow(img_resized)
        ax1.set_title(f"Input RGB ({h}×{w})", fontsize=11, fontweight='bold')
        ax1.axis('off')

        # Plot Ground Truth Depth
        ax2 = plt.subplot(len(test_sizes), 4, idx * 4 + 2)
        im2 = ax2.imshow(depth_gt_resized, cmap='magma', vmin=0, vmax=1)
        ax2.set_title(f"Ground Truth Depth", fontsize=11, fontweight='bold')
        ax2.axis('off')

        # Plot Predicted Depth
        ax3 = plt.subplot(len(test_sizes), 4, idx * 4 + 3)
        im3 = ax3.imshow(depth_pred, cmap='magma', vmin=0, vmax=1)
        ax3.set_title(f"Predicted (MAE: {mae:.3f})", fontsize=11, fontweight='bold')
        ax3.axis('off')

        # Plot Error Map
        ax4 = plt.subplot(len(test_sizes), 4, idx * 4 + 4)
        error_map = np.abs(depth_pred - depth_gt_resized)
        im4 = ax4.imshow(error_map, cmap='hot', vmin=0, vmax=0.3)
        ax4.set_title(f"Error Map (RMSE: {rmse:.3f})", fontsize=11, fontweight='bold')
        ax4.axis('off')
        plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

        print(f"✓ {h}×{w}: MAE={mae:.4f}, RMSE={rmse:.4f}, "
              f"Pred range [{float(depth_pred.min()):.3f}, {float(depth_pred.max()):.3f}]")

    plt.suptitle(
        f"Variable-Size ViT: Ground Truth vs Predictions at Multiple Resolutions\n"
        f"Trained on {config['train_size']}×{config['train_size']}",
        fontsize=14,
        fontweight='bold'
    )
    plt.tight_layout()

    # Save output
    os.makedirs("outputs", exist_ok=True)
    output_path = os.path.join("outputs", f"multires_comparison_{config['num_layers']}_layers.png")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Comparison results saved to: {output_path}")


def test_extreme_resolutions(config: Dict, test_image_path: str):
    """Test model on extreme resolutions to show robustness."""
    print(f"\n{'='*60}")
    print(f"🔬 Extreme Resolution Test")
    print(f"{'='*60}")

    # Load model
    with open(config['checkpoint_path'], 'rb') as f:
        model_state = pickle.load(f)

    rngs = nnx.Rngs(config['seed'])
    model = VariableSizeViTForDepth(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads'],
        default_size=config['train_size'],
        rngs=rngs
    )
    nnx.update(model, model_state)

    # Load test image
    img_original = cv2.cvtColor(cv2.imread(test_image_path), cv2.COLOR_BGR2RGB)
    img_original = img_original.astype(np.float32) / 255.0

    # Test extreme cases
    extreme_sizes = [
        (160, 160),   # Smaller than training
        (768, 768),   # Much larger
        (256, 512),   # 1:2 aspect ratio
        (640, 320),   # 2:1 aspect ratio
    ]

    for h, w in extreme_sizes:
        try:
            img_resized = cv2.resize(img_original, (w, h))
            img_batch = jnp.expand_dims(img_resized, 0)

            # Time the inference
            import time
            start = time.time()
            depth_pred = model(img_batch, train=False)
            elapsed = time.time() - start

            print(f"✓ {h}×{w} ({h*w/1000:.1f}K pixels): {elapsed:.3f}s, "
                  f"Output shape: {depth_pred.shape}")

        except Exception as e:
            print(f"✗ {h}×{w} failed: {str(e)}")


# ============================================================================
# SECTION 5: MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function."""
    print("\n" + "="*60)
    print(" " * 10 + "VARIABLE-SIZE VISION TRANSFORMER")
    print(" " * 15 + "FOR DEPTH ESTIMATION")
    print(" " * 8 + "Train Once, Infer at Any Resolution!")
    print("="*60)

    # Configuration
    config = {
        'seed': 42,
        'data_dir': "./data",
        'batch_size': 8,
        'steps': 3000,  # Reduced for demo
        'lr': 3e-4,
        'num_layers': 6,
        'embed_dim': 192,
        'num_heads': 3,
        'train_size': 224,  # Training resolution
        'checkpoint_path': "./checkpoints/variable_size_vit_6_layers.pkl"
    }

    # Train model (on fixed 224×224)
    final_checkpoint, losses = train_model(config)

    if final_checkpoint:
        # Test on multiple resolutions
        test_image = "./data/input_frames/frame_0014.png"

        # Run detailed single image comparison
        run_single_inference_comparison(config, test_image)

        # Run multi-resolution inference with ground truth
        run_multires_inference(config, test_image)

        # Test extreme resolutions
        test_extreme_resolutions(config, test_image)

    # Plot training curve
    if losses:
        plt.figure(figsize=(10, 6))

        # Plot raw and smoothed loss
        plt.plot(losses, linewidth=1, color='lightblue', alpha=0.5, label='Raw Loss')

        # Smooth the curve
        window_size = 50
        if len(losses) > window_size:
            smoothed = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
            x_smooth = np.arange(window_size//2, len(losses) - window_size//2 + 1)
            plt.plot(x_smooth, smoothed, linewidth=2.5, color='#45B7D1', label='Smoothed Loss')

        plt.title("Variable-Size ViT Training Loss", fontsize=16, fontweight='bold')
        plt.xlabel("Training Step", fontsize=12)
        plt.ylabel("Combined Loss (L1 + Gradient)", fontsize=12)
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Add final loss annotation
        plt.annotate(f'Final: {losses[-1]:.4f}',
                    xy=(len(losses)-1, losses[-1]),
                    xytext=(len(losses)*0.8, losses[-1]*1.2),
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=10, fontweight='bold')

        os.makedirs("outputs", exist_ok=True)
        plt.savefig("outputs/variable_vit_training_loss.png", dpi=150, bbox_inches='tight')
        plt.show()

    print("\n" + "="*60)
    print(" " * 15 + "✨ EXPERIMENT COMPLETE!")
    print(" " * 8 + "Model works at ANY resolution!")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

Writing /content/vision_transformer_depth/anyRES_nnx_string3d_experiment_pipeline.py


In [ ]:
# @title ### training cell
# Change to the project directory
%cd /content/vision_transformer_depth

# Run the full experiment pipeline
!python anyRES_nnx_string3d_experiment_pipeline.py

/content/vision_transformer_depth
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(

          VARIABLE-SIZE VISION TRANSFORMER
               FOR DEPTH ESTIMATION
        Train Once, Infer at Any Resolution!

🚀 Starting Training: Variable-Size 6-Layer Model
   Training on 224×224
   Model supports inference at ANY resolution!
🔥 Training Variable-Size Model (6 Layers):   3% 96/3000 [00:59<01:20, 36.07it/s, loss=0.2300]
[Step 100] Average Loss: 0.2179
🔥 Training Variable-Size Model (6 Layers):   7% 196/3000 [01:01<01:04, 43.24it/s, loss=0.1584]
[Step 200] Average Loss: 0.1869
🔥 Training Variable-Size Model (6 Layers):  10% 296/3000 [01:04<01:02, 43.05it/s